# Full reproduction notebook: Survey-Aware Contrast Set Mining of Diabetes Multimorbidity Groups (BRFSS 2024)

This notebook holds **all the code** behind every number in the paper. The dataset is **not** included.

**What you need**
- The raw CDC BRFSS 2024 file as CSV (`brfss_survey_data_2024.csv`, 457,670 rows). It needs these columns: `_PSU, _STSTR, _LLCPWT, GENHLTH, MENTHLTH, CVDINFR4, CVDCRHD4, CVDSTRK3, ASTHMA3, CHCSCNC1, CHCOCNC1, CHCCOPD3, ADDEPEV3, CHCKDNY2, HAVARTH4, DIABETE4, EDUCA, _TOTINDA, _SEX, _AGEG5YR, _BMI5CAT, _INCOMG1, _SMOKER3`.
- Python 3.10 or newer with `numpy pandas scipy statsmodels matplotlib scikit-learn`.

**How to use it**
1. Set `RAW_CSV` to the **full path** of your file in the first code cell to the path of your file.
2. Run all cells from top to bottom (Kernel → Restart & Run All).
3. The last section prints a table that compares **your** numbers with the numbers in the paper, with PASS or FAIL on each row.

**Run time** (one CPU core, measured in my run): everything except the bootstrap takes about 2 to 4 minutes. The 200-sample bootstrap took about 34 minutes. To test quickly, set `BOOT_B = 10`. The bootstrap rows will then not match, because the paper uses 200 samples.

**How the code is organised:** cells starting with `%%writefile` save each script to disk, exactly as I ran them. Later cells run those scripts. Every result is written to the `results7/` folder (seven groups, the main analysis) and the `results6/` folder (the old six-group version, used only for one comparison). Random seed: 2026.

**Section 12 (new)** holds the checks added after the faculty review: the permutation test behind the Perm. column of Table VI, the R2 (Breslow-Day) sensitivity, the reweighting check for missing data, the null-rate calibration and the cross-check in R's `survey` package.

**Important:** run the SETTINGS cell first. Every later cell needs it. If you restart the kernel, run it again.

In [ ]:
# ===== SETTINGS: change these two lines =====
RAW_CSV  = r"brfss_survey_data_2024.csv"   # path to the raw BRFSS 2024 CSV
WORK_DIR = r"csm_reproduction"             # all outputs go here
BOOT_B   = 200                             # paper uses 200 bootstrap samples
RUN_SIX_GROUP_COMPARISON = True            # reproduces the "six vs seven groups" numbers

import os, sys, subprocess, time, json
RAW_CSV = os.path.abspath(RAW_CSV)
if not os.path.exists(RAW_CSV):
    raise FileNotFoundError(f"Raw file not found: {RAW_CSV}  Put the FULL path of the CSV in RAW_CSV.")
os.makedirs(WORK_DIR, exist_ok=True); os.chdir(WORK_DIR)
print("Working folder:", os.getcwd())

import numpy, pandas, scipy, statsmodels, matplotlib
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| statsmodels", statsmodels.__version__, "| matplotlib", matplotlib.__version__)
print("Versions used for the paper: Python 3.13, numpy 2.5.3, pandas 3.0.5, scipy 1.18.1, statsmodels 0.15.0, matplotlib 3.11.2")

def run_script(script, *args):
    """Run a script with this Python and show its output live."""
    t0 = time.time()
    p = subprocess.Popen([sys.executable, script, *map(str, args)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    for line in p.stdout: print(line, end="")
    if p.wait() != 0: raise RuntimeError(f"{script} failed")
    print(f"--- {script} finished in {time.time()-t0:.1f} s")
    return time.time() - t0

print("Settings OK. Raw file found:", RAW_CSV)

## 1. Save the scripts
### 1.1 `build_data.py`: builds the analysis set from the raw file
Keeps adults with diagnosed diabetes (`DIABETE4 == 1`), builds the five condition flags and the groups, and recodes the eight traits. Writes a *legacy* version (keeps BRFSS 'don't know/refused' codes, as in the original preprocessing) and the *corrected* version used in the paper (those codes treated as missing).

In [ ]:
%%writefile build_data.py
"""Rebuild the diabetes multimorbidity analysis set from raw BRFSS 2024.
legacy=True reproduces the original mining_ready_data.csv (keeps smoker=9 and age=14,
which are BRFSS 'don't know/refused' codes). legacy=False treats them as missing."""
import sys, numpy as np, pandas as pd
RAW = sys.argv[1]
COLS=["_PSU","_STSTR","_LLCPWT","GENHLTH","MENTHLTH","CVDINFR4","CVDCRHD4","CVDSTRK3","ASTHMA3","CHCSCNC1",
      "CHCOCNC1","CHCCOPD3","ADDEPEV3","CHCKDNY2","HAVARTH4","DIABETE4","EDUCA","_TOTINDA","_SEX","_AGEG5YR",
      "_BMI5CAT","_INCOMG1","_SMOKER3"]
A=["bmi_cat","smoker_status","physical_activity","general_health","mh_days_cat","income_group","education","age_group"]
def build(legacy):
    d=pd.read_csv(RAW,usecols=COLS,low_memory=False)
    d=d[d.DIABETE4==1].copy()                       # diagnosed diabetes only (excl. gestational, prediabetes)
    y=lambda c:(d[c]==1)
    d["cond_cvd"]=(y("CVDINFR4")|y("CVDCRHD4")|y("CVDSTRK3")).astype(int)
    d["cond_ckd"]=y("CHCKDNY2").astype(int); d["cond_arthritis"]=y("HAVARTH4").astype(int)
    d["cond_copd"]=y("CHCCOPD3").astype(int); d["cond_depression"]=y("ADDEPEV3").astype(int)
    d["cond_asthma"]=y("ASTHMA3").astype(int); d["cond_cancer"]=(y("CHCSCNC1")|y("CHCOCNC1")).astype(int)
    cnt=d[["cond_cvd","cond_ckd","cond_arthritis","cond_copd","cond_depression"]].sum(axis=1)
    d["n_conditions"]=cnt
    d["mm_group"]=np.select([cnt==0,cnt>=2,(cnt==1)&(d.cond_cvd==1),(cnt==1)&(d.cond_ckd==1),(cnt==1)&(d.cond_arthritis==1)],
        ["diabetes_only","diabetes_multi_2plus","diabetes_cvd","diabetes_ckd","diabetes_arthritis"],"diabetes_other_single")
    m=d.MENTHLTH
    d["mh_days_cat"]=pd.Series(np.select([m==88,(m>=1)&(m<=5),(m>=6)&(m<=13),(m>=14)&(m<=30)],
        ["none","1-5 days","6-13 days","14-30 days"],"NA"),index=d.index).replace("NA",np.nan)
    k=lambda s,ok:s.where(s.isin(ok))
    d["general_health"]=k(d.GENHLTH,[1,2,3,4,5]); d["physical_activity"]=k(d._TOTINDA,[1,2])
    d["age_group"]=k(d._AGEG5YR,range(1,15) if legacy else range(1,14))
    d["smoker_status"]=k(d._SMOKER3,[1,2,3,4,9] if legacy else [1,2,3,4])
    d["income_group"]=k(d._INCOMG1,range(1,8)); d["bmi_cat"]=k(d._BMI5CAT,[1,2,3,4]); d["education"]=k(d.EDUCA,range(1,7))
    return d, A
if __name__=="__main__":
    for legacy,name in [(True,"legacy"),(False,"corrected")]:
        d,A=build(legacy)
        d.to_csv(f"diabetes_all_{name}.csv",index=False)
        ok=d[A].notna().all(axis=1)
        d[ok].to_csv(f"analysis_{name}.csv",index=False)
        print(name, "diabetes:",len(d),"analysis:",ok.sum())


### 1.2 `csm_upgrade.py`: the main pipeline
Candidate generation, survey-weighted supports, the design-based (Rao-Scott) test, Holm correction, the Δ ≥ 0.10 filter, pruning rules R1 and R2, one-vs-rest differences with 95% CIs, threshold grid, baselines (multinomial logit, WRAcc), Cramér's V, age strata and the run without outcome traits.

In [ ]:
%%writefile csm_upgrade.py
#!/usr/bin/env python3
"""
Upgrade experiments for:
  "Contrast Set Mining for Discovering Distinctive Multimorbidity Patterns in
   Diabetes-Related Chronic Conditions Using CDC BRFSS Health Indicators"

What this script does (one section per experiment, E0 to E10):
  E0  Group sizes for the analysis set + check of who was dropped (missing data)
  E1  Survey-weighted supports + design-based tests (Wald F and first-order
      Rao-Scott), side by side with the unweighted analysis
  E2  One-vs-rest contrasts: support difference, risk ratio, odds ratio,
      95% CIs, and WRAcc
  E3  Explicit redundancy pruning (complement rule + STUCCO-style
      "surprising" rule for two-item patterns); reports which pairs survive
  E4  Threshold sensitivity (delta x minimum count)
  E5  Bootstrap stability (PSU bootstrap within strata, B = 200)
  E6  Baselines: multinomial logistic regression and WRAcc subgroup discovery
  E7  Cramer's V with correct Cohen labels (uses df* = min(r, c) - 1)
  E8  Circularity check (which conditions define each group) and a rerun
      without outcome-type attributes
  E9  Age-stratified mining with the SAME pipeline (age_group removed)
  E10 Condition-specific contrast among the four single-condition groups
  FIG All figures with readable labels

Usage:
  python csm_upgrade.py --data analysis_ready.csv --full diabetes_all.csv --out results

  --data : the 49,757-row analysis set (one row per respondent, columns below)
  --full : (optional) the 65,809-row diabetes file BEFORE dropping missing
           values; needed only for E0's dropout check.

Required columns (rename in CFG if yours differ):
  mm_group, the 8 mining attributes, _LLCPWT, _STSTR, _PSU
Optional columns: _SEX, and condition flags listed in CFG["conditions"].

Every number this script writes comes from your data. Nothing is hard-coded.
Random seed is fixed (SEED) so every run gives the same output.
"""
import argparse
import itertools
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
SEED = 2026

# --------------------------------------------------------------------------
# CONFIGURATION. Check every entry against your own preprocessing code.
# --------------------------------------------------------------------------
CFG = dict(
    group="mm_group",
    groups=["diabetes_only", "diabetes_cvd", "diabetes_ckd",
            "diabetes_arthritis", "diabetes_other_single",
            "diabetes_multi_2plus"],
    single_condition_groups=["diabetes_cvd", "diabetes_ckd",
                             "diabetes_arthritis", "diabetes_other_single"],
    attrs=["mh_days_cat", "general_health", "physical_activity", "age_group",
           "smoker_status", "income_group", "bmi_cat", "education"],
    # Attributes that are closer to health OUTCOMES than to risk factors.
    # E8 reruns the mining without them.
    outcome_attrs=["mh_days_cat", "general_health"],
    weight="_LLCPWT", strata="_STSTR", psu="_PSU", sex="_SEX",
    age_attr="age_group",
    under65_codes=[1, 2, 3, 4, 5, 6, 7, 8, 9],   # _AGEG5YR 1..9 = 18 to 64
    # 0/1 flags for each chronic condition used to build the groups (E8).
    # Put the column names of YOUR processed file here.
    conditions={
        "CVD (heart attack, CHD or stroke)": "cond_cvd",
        "Kidney disease": "cond_ckd",
        "Arthritis": "cond_arthritis",
        "Depressive disorder": "cond_depression",
        "COPD": "cond_copd",
        "Asthma (not used for groups)": "cond_asthma",
        "Cancer (not used for groups)": "cond_cancer",
    },
    min_count=30, delta=0.10, alpha=0.05,
    surprise_eps=0.02,       # E3: min. |observed - expected| support gap
    boot_B=200,
    sens_delta=[0.05, 0.10, 0.15], sens_min_count=[30, 100, 500],
)

# Readable labels. Codes follow the 2024 BRFSS codebook for GENHLTH,
# _TOTINDA, _SMOKER3, _BMI5CAT, _AGEG5YR, EDUCA and _INCOMG1.
# VERIFY each map against the codebook and your own recoding.
LABELS = {
    "general_health": {1: "Excellent", 2: "Very good", 3: "Good", 4: "Fair", 5: "Poor"},
    "physical_activity": {1: "Any leisure activity", 2: "No leisure activity"},
    "smoker_status": {1: "Current, daily", 2: "Current, some days",
                      3: "Former smoker", 4: "Never smoked"},
    "bmi_cat": {1: "Underweight", 2: "Normal weight", 3: "Overweight", 4: "Obese"},
    "age_group": {1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39", 5: "40-44",
                  6: "45-49", 7: "50-54", 8: "55-59", 9: "60-64", 10: "65-69",
                  11: "70-74", 12: "75-79", 13: "80+"},
    "education": {1: "No school", 2: "Grades 1-8", 3: "Grades 9-11",
                  4: "High school", 5: "Some college", 6: "College graduate"},
    "income_group": {1: "<$15k", 2: "$15k-25k", 3: "$25k-35k", 4: "$35k-50k",
                     5: "$50k-100k", 6: "$100k-200k", 7: "$200k+"},
    "mh_days_cat": {"none": "0 days", "1-5 days": "1-5 days",
                    "6-13 days": "6-13 days", "14-30 days": "14-30 days"},
}
ATTR_NAMES = {"mh_days_cat": "Poor mental health days",
              "general_health": "General health",
              "physical_activity": "Physical activity",
              "age_group": "Age", "smoker_status": "Smoking",
              "income_group": "Income", "bmi_cat": "BMI",
              "education": "Education"}
GROUP_NAMES = {"diabetes_only": "Diabetes only", "diabetes_cvd": "Diab.+CVD",
               "diabetes_ckd": "Diab.+CKD", "diabetes_arthritis": "Diab.+arthritis",
               "diabetes_other_single": "Diab.+other single",
               "diabetes_multi_2plus": "Diab.+2 or more"}


def norm_code(v):
    """Turn 4.0 / np.float64(4.0) / '4' into 4; keep text codes as text."""
    try:
        f = float(v)
        return int(f) if f.is_integer() else f
    except (TypeError, ValueError):
        return v


def item_label(attr, val):
    v = norm_code(val)
    lab = LABELS.get(attr, {}).get(v, str(v))
    return f"{ATTR_NAMES.get(attr, attr)}: {lab}"


def pattern_label(items):
    return " AND ".join(item_label(a, v) for a, v in items)


# --------------------------------------------------------------------------
# Survey helpers (Taylor linearization, stratified with-replacement PSUs)
# --------------------------------------------------------------------------
class Design:
    """Holds weights, strata and PSU ids; computes linearized covariances."""

    def __init__(self, df, cfg, weighted=True):
        n = len(df)
        self.w = df[cfg["weight"]].to_numpy(float) if weighted else np.ones(n)
        if weighted:
            st = df[cfg["strata"]].astype(str).to_numpy()
            ps = df[cfg["psu"]].astype(str).to_numpy()
        else:  # simple random sampling: each row is its own PSU, one stratum
            st = np.zeros(n, dtype=int).astype(str)
            ps = np.arange(n).astype(str)
        key = pd.Series(st) + "|" + pd.Series(ps)
        self.psu_idx, psu_keys = pd.factorize(key)
        psu_stratum = pd.Series(psu_keys).str.split("|").str[0]
        self.str_of_psu, _ = pd.factorize(psu_stratum)
        self.n_psu = len(psu_keys)
        self.n_str = self.str_of_psu.max() + 1
        nh = np.bincount(self.str_of_psu, minlength=self.n_str)
        self.nh = nh
        self.df_design = self.n_psu - self.n_str   # design degrees of freedom
        self.weighted = weighted

    def cov_of_totals(self, U):
        """U: n x m matrix of influence values. Returns m x m covariance."""
        m = U.shape[1]
        T = np.zeros((self.n_psu, m))
        np.add.at(T, self.psu_idx, U)                       # PSU totals
        S = np.zeros((self.n_str, m))
        np.add.at(S, self.str_of_psu, T)
        mean_h = S / self.nh[:, None]
        D = T - mean_h[self.str_of_psu]                     # centred in stratum
        f = np.where(self.nh > 1, self.nh / np.maximum(self.nh - 1, 1), 0.0)
        D = D * np.sqrt(f[self.str_of_psu])[:, None]        # lonely PSU -> 0
        return D.T @ D


def domain_props(Y, g, design, levels):
    """Weighted proportions of the columns of Y within each domain level, with
    their joint linearized covariance. Y: n x J (0/1). Returns p (K*J,), V."""
    w = design.w
    K, J = len(levels), Y.shape[1]
    P = np.zeros((K, J))
    U = np.zeros((len(g), K * J))
    for k, lev in enumerate(levels):
        d = (g == lev)
        Wk = w[d].sum()
        P[k] = (w[d, None] * Y[d]).sum(0) / Wk
        U[:, k * J:(k + 1) * J] = (d * w)[:, None] * (Y - P[k]) / Wk
    return P.ravel(), design.cov_of_totals(U), P


def wald_homogeneity(Y, g, design, levels):
    """Design-based test that the distribution of Y is the same in all
    groups. Returns Wald F, its p-value, first-order Rao-Scott chi2 and p."""
    p, V, P = domain_props(Y, g, design, levels)
    K, J = P.shape
    # contrasts: p_{k,j} - p_{K,j}, k < K
    C = np.zeros(((K - 1) * J, K * J))
    for k in range(K - 1):
        for j in range(J):
            C[k * J + j, k * J + j] = 1
            C[k * J + j, (K - 1) * J + j] = -1
    q = C.shape[0]
    cp = C @ p
    CVC = C @ V @ C.T
    W = float(cp @ np.linalg.pinv(CVC) @ cp)
    d = max(design.df_design, q + 1)
    F = W * (d - q + 1) / (d * q)
    pF = stats.f.sf(F, q, d - q + 1)
    # SRS covariance (multinomial within each domain, unweighted n_k)
    Vs = np.zeros((K * J, K * J))
    for k, lev in enumerate(levels):
        nk = (g == lev).sum()
        pk = P[k]
        Vs[k * J:(k + 1) * J, k * J:(k + 1) * J] = (np.diag(pk) - np.outer(pk, pk)) / nk
    CVsC = C @ Vs @ C.T
    Ws = float(cp @ np.linalg.pinv(CVsC) @ cp)           # ~ Pearson X2
    deff_mean = np.trace(np.linalg.pinv(CVsC) @ CVC) / q
    rs1 = Ws / max(deff_mean, 1e-12)
    prs = stats.chi2.sf(rs1, q)
    return dict(wald_F=F, p_wald=pF, rs1_chi2=rs1, p_rs1=prs,
                pearson_w=Ws, mean_deff=deff_mean, df=q)


# --------------------------------------------------------------------------
# Candidate generation and counting
# --------------------------------------------------------------------------
def build_items(df, attrs):
    items = []
    for a in attrs:
        for v in sorted(df[a].dropna().unique(), key=lambda x: str(x)):
            items.append((a, v))
    return items


def mask_of(df, items, cache):
    m = None
    for it in items:
        if it not in cache:
            cache[it] = (df[it[0]] == it[1]).to_numpy()
        m = cache[it] if m is None else (m & cache[it])
    return m


def candidates(items, max_len=2):
    cands = [(it,) for it in items]
    if max_len >= 2:
        for a, b in itertools.combinations(items, 2):
            if a[0] != b[0]:
                cands.append((a, b))
    return cands


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * p[i])
        adj[i] = min(run, 1.0)
    return adj


def mine(df, cfg, attrs=None, groups=None, weighted=False, delta=None,
         min_count=None, max_len=2, design=None, test="auto"):
    """STUCCO-style mining. Returns a table with one row per candidate that
    passed the minimum count, with supports, Delta, max/min group, p, Holm p."""
    attrs = attrs or cfg["attrs"]
    groups = groups or cfg["groups"]
    delta = cfg["delta"] if delta is None else delta
    min_count = cfg["min_count"] if min_count is None else min_count
    df = df[df[cfg["group"]].isin(groups)].reset_index(drop=True)
    g = df[cfg["group"]].to_numpy()
    gidx = pd.Categorical(g, categories=groups).codes
    K = len(groups)
    w = df[cfg["weight"]].to_numpy(float) if weighted else np.ones(len(df))
    n_g = np.bincount(gidx, minlength=K)
    W_g = np.bincount(gidx, weights=w, minlength=K)
    if weighted and design is None:
        design = Design(df, cfg, weighted=True)
    cache = {}
    rows = []
    items = build_items(df, attrs)
    n_cand_total = 0
    for c in candidates(items, max_len):
        n_cand_total += 1
        m = mask_of(df, c, cache)
        cnt = m.sum()
        if cnt < min_count:
            continue
        x_g = np.bincount(gidx[m], minlength=K)
        s = np.bincount(gidx[m], weights=w[m], minlength=K) / W_g
        if weighted and test != "chi2":
            t = wald_homogeneity(m[:, None].astype(float), g, design, groups)
            p = t["p_rs1"]
        else:
            tab = np.vstack([x_g, n_g - x_g])
            p = stats.chi2_contingency(tab, correction=False)[1] if (tab.min(1) > 0).all() else 1.0
        row = dict(pattern=c, length=len(c), label=pattern_label(c), count=int(cnt),
                   delta=s.max() - s.min(), max_group=groups[int(s.argmax())],
                   min_group=groups[int(s.argmin())], p=p)
        for k, gr in enumerate(groups):
            row["supp_" + gr] = s[k]
        rows.append(row)
    res = pd.DataFrame(rows)
    res.attrs["n_candidates_generated"] = n_cand_total
    if len(res):
        res["p_holm"] = holm(res["p"].to_numpy())
        res["significant"] = (res["p_holm"] < cfg["alpha"]) & (res["delta"] >= delta)
    return res


# --------------------------------------------------------------------------
# E3. Redundancy pruning (explicit rules)
# --------------------------------------------------------------------------
def prune(res, df, cfg, groups=None, weighted=False):
    """Rule R1 (complement): for a binary attribute, A=a and A=b carry the same
    contrast; keep only the level with the larger Delta (ties: first code).
    Rule R2 (STUCCO-style surprise, Bay & Pazzani 2001): a two-item pattern
    A AND B is kept only if (i) its group supports are significantly different
    from what A and B predict if they were independent within each group
    (Breslow-Day test of equal A-B odds ratios across groups, Holm-corrected),
    and (ii) max_g |s_obs(g) - s(A|g)s(B|g)| >= surprise_eps.
    Single-item patterns are only subject to R1."""
    from statsmodels.stats.contingency_tables import StratifiedTable
    groups = groups or cfg["groups"]
    sig = res[res["significant"]].copy()
    sig["drop_reason"] = ""
    d = df[df[cfg["group"]].isin(groups)].reset_index(drop=True)
    gcol = d[cfg["group"]].to_numpy()
    w = d[cfg["weight"]].to_numpy(float) if weighted else np.ones(len(d))
    # R1 complements
    singles = sig[sig["length"] == 1]
    for a in singles["pattern"].map(lambda c: c[0][0]).unique():
        if d[a].nunique() == 2:
            rows = singles[singles["pattern"].map(lambda c: c[0][0] == a)]
            if len(rows) == 2:
                drop = rows.sort_values("delta", ascending=False).index[1]
                sig.loc[drop, "drop_reason"] = "R1 complement of a kept item"
    # R2 surprise test for pairs
    pairs = sig[sig["length"] == 2]
    pvals, surpr, idx = [], [], []
    for i, r in pairs.iterrows():
        (a, b) = r["pattern"]
        # Pairs are tested even when a parent is not a contrast on its own:
        # a pure interaction is exactly what STUCCO wants to keep.
        A = (d[a[0]] == a[1]).to_numpy()
        B = (d[b[0]] == b[1]).to_numpy()
        tabs, gap = [], 0.0
        for gr in groups:
            k = gcol == gr
            wk = w[k]
            Ak, Bk = A[k], B[k]
            t = np.array([[np.sum(Ak & Bk), np.sum(Ak & ~Bk)],
                          [np.sum(~Ak & Bk), np.sum(~Ak & ~Bk)]], float) + 0.5
            tabs.append(t)
            sA = wk[Ak].sum() / wk.sum()
            sB = wk[Bk].sum() / wk.sum()
            sAB = wk[Ak & Bk].sum() / wk.sum()
            gap = max(gap, abs(sAB - sA * sB))
        st = StratifiedTable(tabs)
        pvals.append(st.test_equal_odds().pvalue)
        surpr.append(gap)
        idx.append(i)
    if idx:
        ph = holm(pvals)
        for i, p, s in zip(idx, ph, surpr):
            sig.loc[i, "p_surprise_holm"] = p
            sig.loc[i, "surprise_gap"] = s
            if not (p < cfg["alpha"] and s >= cfg["surprise_eps"]):
                sig.loc[i, "drop_reason"] = "R2 not surprising given parents"
    sig["kept"] = sig["drop_reason"] == ""
    return sig


# --------------------------------------------------------------------------
# E2. One-vs-rest contrasts with CIs and WRAcc
# --------------------------------------------------------------------------
def one_vs_rest(df, cfg, pattern, groups=None, weighted=True, design=None):
    groups = groups or cfg["groups"]
    d = df[df[cfg["group"]].isin(groups)].reset_index(drop=True)
    if design is None:
        design = Design(d, cfg, weighted=weighted)
    m = np.ones(len(d), bool)
    for a, v in pattern:
        m &= (d[a] == v).to_numpy()
    y = m.astype(float)
    g = d[cfg["group"]].to_numpy()
    w = design.w
    N = w.sum()
    out = []
    z = stats.norm.ppf(0.975)
    for gr in groups:
        ind = np.where(g == gr, "in", "rest")
        p, V, P = domain_props(y[:, None], ind, design, ["in", "rest"])
        s1, s0 = p[0], p[1]
        diff = s1 - s0
        se_d = np.sqrt(max(V[0, 0] + V[1, 1] - 2 * V[0, 1], 0))
        rr = s1 / s0 if s0 > 0 else np.nan
        grad = np.array([1 / s1, -1 / s0]) if s1 > 0 and s0 > 0 else np.array([np.nan, np.nan])
        se_lrr = np.sqrt(max(grad @ V @ grad, 0))
        orr = (s1 / (1 - s1)) / (s0 / (1 - s0)) if 0 < s1 < 1 and 0 < s0 < 1 else np.nan
        go = np.array([1 / (s1 * (1 - s1)), -1 / (s0 * (1 - s0))]) if np.isfinite(orr) else grad
        se_lor = np.sqrt(max(go @ V @ go, 0))
        cov = w[m].sum() / N
        prec = w[m & (g == gr)].sum() / max(w[m].sum(), 1e-12)
        prior = w[g == gr].sum() / N
        out.append(dict(pattern=pattern_label(pattern), group=gr,
                        supp_group=s1, supp_rest=s0,
                        diff=diff, diff_lo=diff - z * se_d, diff_hi=diff + z * se_d,
                        rr=rr, rr_lo=np.exp(np.log(rr) - z * se_lrr), rr_hi=np.exp(np.log(rr) + z * se_lrr),
                        or_=orr, or_lo=np.exp(np.log(orr) - z * se_lor), or_hi=np.exp(np.log(orr) + z * se_lor),
                        wracc=cov * (prec - prior)))
    return pd.DataFrame(out)


# --------------------------------------------------------------------------
# E7. Cramer's V with Cohen labels
# --------------------------------------------------------------------------
def cohen_label(v, dfstar):
    s, m, l = 0.10 / np.sqrt(dfstar), 0.30 / np.sqrt(dfstar), 0.50 / np.sqrt(dfstar)
    return "negligible" if v < s else "small" if v < m else "medium" if v < l else "large"


def cramers_table(df, cfg, design_w, design_u):
    rows = []
    g = df[cfg["group"]].to_numpy()
    for a in cfg["attrs"]:
        levels = sorted(df[a].dropna().unique(), key=str)
        Y = np.column_stack([(df[a] == v).to_numpy(float) for v in levels[:-1]])
        tab = pd.crosstab(df[a], df[cfg["group"]]).to_numpy()
        chi2, p, dof, _ = stats.chi2_contingency(tab, correction=False)
        n = tab.sum()
        r, c = tab.shape
        dfstar = min(r, c) - 1
        V = np.sqrt(chi2 / (n * dfstar))
        tw = wald_homogeneity(Y, g, design_w, cfg["groups"])
        Vw = np.sqrt(tw["pearson_w"] / (n * dfstar))  # weighted proportions
        rows.append(dict(attribute=ATTR_NAMES[a], levels=r, df_star=dfstar,
                         V_unweighted=V, chi2=chi2, p=p, label_unweighted=cohen_label(V, dfstar),
                         V_weighted=Vw, rao_scott_chi2=tw["rs1_chi2"], p_rao_scott=tw["p_rs1"],
                         wald_F=tw["wald_F"], p_wald=tw["p_wald"], mean_deff=tw["mean_deff"],
                         label_weighted=cohen_label(Vw, dfstar),
                         small_cut=0.10 / np.sqrt(dfstar), medium_cut=0.30 / np.sqrt(dfstar)))
    return pd.DataFrame(rows).sort_values("V_unweighted", ascending=False)


# --------------------------------------------------------------------------
# E5. Bootstrap
# --------------------------------------------------------------------------
class PSUBootstrap:
    """Resample PSUs with replacement within each stratum (n_h draws per
    stratum), keeping all rows of a drawn PSU. Unweighted: row bootstrap."""

    def __init__(self, df, cfg, weighted):
        self.n = len(df)
        self.weighted = weighted
        if not weighted:
            return
        d = Design(df, cfg, weighted=True)
        order = np.argsort(d.psu_idx, kind="stable")             # rows by PSU
        counts = np.bincount(d.psu_idx, minlength=d.n_psu)
        self.row_order = order
        self.row_start = np.concatenate([[0], np.cumsum(counts)[:-1]])
        self.row_count = counts
        psu_by_str = np.argsort(d.str_of_psu, kind="stable")     # PSUs by stratum
        self.psu_by_str = psu_by_str
        self.str_start = np.concatenate([[0], np.cumsum(d.nh)[:-1]])
        self.nh = d.nh
        self.str_of_slot = d.str_of_psu[psu_by_str]

    def draw(self, rng):
        if not self.weighted:
            return rng.integers(0, self.n, self.n)
        h = self.str_of_slot
        pick = self.psu_by_str[self.str_start[h] + (rng.random(len(h)) * self.nh[h]).astype(int)]
        starts, cnts = self.row_start[pick], self.row_count[pick]
        rep = np.repeat(starts - np.concatenate([[0], np.cumsum(cnts)[:-1]]), cnts)
        return self.row_order[np.arange(cnts.sum()) + rep]


def bootstrap_stability(df, cfg, final, weighted, B, rng):
    keys = [tuple(p) for p in final["pattern"]]
    found = {k: 0 for k in keys}
    same_max = {k: 0 for k in keys}
    max_ref = dict(zip(keys, final["max_group"]))
    sampler = PSUBootstrap(df, cfg, weighted)
    t0 = time.time()
    for b in range(B):
        idx = sampler.draw(rng)
        db = df.iloc[idx].reset_index(drop=True)
        r = mine(db, cfg, weighted=weighted, test="chi2")   # chi2 for speed
        s = r[r["significant"]]
        smap = dict(zip(s["pattern"].map(tuple), s["max_group"]))
        for k in keys:
            if k in smap:
                found[k] += 1
                same_max[k] += (smap[k] == max_ref[k])
        if b == 0:
            print(f"    one bootstrap replicate took {time.time()-t0:.1f}s")
    return pd.DataFrame(dict(pattern=[pattern_label(k) for k in keys],
                             rediscovery_rate=[found[k] / B for k in keys],
                             same_max_group_rate=[same_max[k] / B for k in keys]))


# --------------------------------------------------------------------------
# E6. Baselines
# --------------------------------------------------------------------------
def baseline_multinomial(df, cfg):
    """Multinomial logit of group on all attributes (one-hot, first level as
    reference). Unweighted fit for SEs; weighted point estimates from sklearn.
    Reports odds ratios vs diabetes_only."""
    import statsmodels.api as sm
    X = pd.get_dummies(df[cfg["attrs"]].astype(str), drop_first=True, dtype=float)
    y = pd.Categorical(df[cfg["group"]], categories=cfg["groups"]).codes
    model = sm.MNLogit(y, sm.add_constant(X)).fit(disp=False, maxiter=200, method="newton")
    rows = []
    for k in range(1, len(cfg["groups"])):
        par = model.params[k - 1]
        se = model.bse[k - 1]
        for name in X.columns:
            a, v = name.rsplit("_", 1)
            rows.append(dict(group=cfg["groups"][k], term=item_label(a, v),
                             OR=np.exp(par[name]),
                             lo=np.exp(par[name] - 1.96 * se[name]),
                             hi=np.exp(par[name] + 1.96 * se[name]),
                             p=model.pvalues[k - 1][name]))
    try:
        from sklearn.linear_model import LogisticRegression
        lr = LogisticRegression(max_iter=2000, C=1e6)
        lr.fit(X.to_numpy(), y, sample_weight=df[cfg["weight"]].to_numpy(float))
        ref = lr.coef_[0]
        Wt = {}
        for k in range(1, len(cfg["groups"])):
            for j, name in enumerate(X.columns):
                a, v = name.rsplit("_", 1)
                Wt[(cfg["groups"][k], item_label(a, v))] = np.exp(lr.coef_[k][j] - ref[j])
        for r in rows:
            r["OR_weighted_point"] = Wt.get((r["group"], r["term"]), np.nan)
    except Exception as e:  # sklearn missing: skip weighted column
        print("    sklearn weighted fit skipped:", e)
    return pd.DataFrame(rows), model.prsquared


def baseline_wracc(df, cfg, top_k=10, weighted=True):
    """Exhaustive subgroup discovery (depth <= 2) with WRAcc per target group.
    Same search space as the contrast miner, so the comparison is fair."""
    d = df.reset_index(drop=True)
    w = d[cfg["weight"]].to_numpy(float) if weighted else np.ones(len(d))
    g = d[cfg["group"]].to_numpy()
    N = w.sum()
    cache = {}
    items = build_items(d, cfg["attrs"])
    cands = candidates(items, 2)
    out = []
    for gr in cfg["groups"]:
        tgt = g == gr
        prior = w[tgt].sum() / N
        rows = []
        for c in cands:
            m = mask_of(d, c, cache)
            if m.sum() < cfg["min_count"]:
                continue
            cov = w[m].sum() / N
            prec = w[m & tgt].sum() / w[m].sum()
            rows.append((c, cov * (prec - prior), prec, cov))
        rows.sort(key=lambda r: -r[1])
        for rank, (c, wr, prec, cov) in enumerate(rows[:top_k], 1):
            out.append(dict(group=gr, rank=rank, pattern=pattern_label(c), length=len(c),
                            wracc=wr, precision=prec, coverage=cov, prior=prior))
    return pd.DataFrame(out)


# --------------------------------------------------------------------------
# E0. Group table and dropout check
# --------------------------------------------------------------------------
def group_table(df, cfg):
    w = df[cfg["weight"]]
    t = df.groupby(cfg["group"]).agg(n=(cfg["group"], "size"))
    t["pct_unweighted"] = 100 * t["n"] / t["n"].sum()
    t["pct_weighted"] = 100 * df.groupby(cfg["group"])[cfg["weight"]].sum() / w.sum()
    return t.reindex(cfg["groups"]).reset_index()


def dropout_check(full, cfg):
    """Compare respondents kept vs dropped (missing in any mining attribute)."""
    kept = full[cfg["attrs"]].notna().all(axis=1)
    full = full.assign(status=np.where(kept, "kept", "dropped"))
    rows = []
    for col in [cfg["group"], cfg["age_attr"], cfg["sex"]]:
        if col not in full:
            continue
        tab = pd.crosstab(full[col], full["status"])
        chi2, p, _, _ = stats.chi2_contingency(tab)
        V = np.sqrt(chi2 / (tab.to_numpy().sum() * (min(tab.shape) - 1)))
        pct = 100 * tab / tab.sum()
        for lev in tab.index:
            rows.append(dict(variable=col, level=str(lev),
                             pct_kept=pct.loc[lev, "kept"], pct_dropped=pct.loc[lev, "dropped"],
                             cramers_v=V, p=p))
    miss = full[cfg["attrs"]].isna().mean().rename("share_missing").reset_index()
    return pd.DataFrame(rows), miss, int((~kept).sum()), int(kept.sum())


# --------------------------------------------------------------------------
# Figures
# --------------------------------------------------------------------------
def fig_heatmap(final, cfg, path, title_suffix=""):
    import matplotlib.pyplot as plt
    supp = final[["supp_" + g for g in cfg["groups"]]].to_numpy()
    fig, ax = plt.subplots(figsize=(7.2, 0.32 * len(final) + 1.4))
    im = ax.imshow(supp, cmap="Greys", vmin=0, vmax=1, aspect="auto")
    for i in range(supp.shape[0]):
        jmax, jmin = supp[i].argmax(), supp[i].argmin()
        for j in range(supp.shape[1]):
            txt = f"{supp[i, j]:.2f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if supp[i, j] > 0.55 else "black",
                    fontweight="bold" if j in (jmax, jmin) else "normal")
        ax.add_patch(plt.Rectangle((jmax - .5, i - .5), 1, 1, fill=False, lw=1.6, ec="black"))
        ax.add_patch(plt.Rectangle((jmin - .5, i - .5), 1, 1, fill=False, lw=1.2, ec="black", ls="--"))
    ax.set_yticks(range(len(final)))
    ax.set_yticklabels(final["label"], fontsize=8)
    ax.set_xticks(range(len(cfg["groups"])))
    ax.set_xticklabels([GROUP_NAMES[g] for g in cfg["groups"]], fontsize=8, rotation=30, ha="right")
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("Support (share of group)", fontsize=8)
    cb.ax.tick_params(labelsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.close(fig)


def fig_forest(ovr, path):
    """One row per final pattern: one-vs-rest support difference with 95% CI
    for the group with the largest |difference|."""
    import matplotlib.pyplot as plt
    best = ovr.loc[ovr.groupby("pattern")["diff"].apply(lambda s: s.abs().idxmax())]
    best = best.sort_values("diff")
    fig, ax = plt.subplots(figsize=(3.5, 0.25 * len(best) + 0.9))
    y = np.arange(len(best))
    ax.errorbar(best["diff"], y, xerr=[best["diff"] - best["diff_lo"], best["diff_hi"] - best["diff"]],
                fmt="o", color="black", ms=3, capsize=2, lw=1)
    ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{p} [{GROUP_NAMES[g]}]" for p, g in zip(best["pattern"], best["group"])], fontsize=6)
    ax.set_xlabel("Group minus rest (95% CI)", fontsize=7)
    ax.tick_params(axis="x", labelsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.close(fig)


def fig_age(r_u65, r_65, path, top=10):
    import matplotlib.pyplot as plt
    a = r_u65.set_index("label")["delta"]
    b = r_65.set_index("label")["delta"]
    labs = (pd.concat([a, b], axis=1).max(axis=1).sort_values(ascending=False).index[:top])
    fig, ax = plt.subplots(figsize=(3.5, 0.28 * len(labs) + 0.9))
    y = np.arange(len(labs))
    ax.barh(y - 0.18, [a.get(l, 0) for l in labs], 0.36, color="0.25", label="Under 65")
    ax.barh(y + 0.18, [b.get(l, 0) for l in labs], 0.36, color="0.7", hatch="///", label="65 and over")
    ax.set_yticks(y)
    ax.set_yticklabels(labs, fontsize=6)
    ax.invert_yaxis()
    ax.set_xlabel(r"$\Delta$ (max minus min group support)", fontsize=7)
    ax.tick_params(axis="x", labelsize=7)
    ax.legend(fontsize=6, frameon=False)
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.close(fig)


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--full", default=None)
    ap.add_argument("--out", default="results")
    ap.add_argument("--boot", type=int, default=CFG["boot_B"])
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    cfg = CFG
    rng = np.random.default_rng(SEED)
    out = lambda f: os.path.join(args.out, f)
    summary = {}

    df = pd.read_csv(args.data, low_memory=False)
    for a in cfg["attrs"]:
        df[a] = df[a].map(norm_code)
    df = df[df[cfg["group"]].isin(cfg["groups"])].reset_index(drop=True)
    print(f"Loaded {len(df):,} rows")
    dw = Design(df, cfg, weighted=True)
    du = Design(df, cfg, weighted=False)
    summary["n_analysis"] = len(df)
    summary["n_psu"], summary["n_strata"] = int(dw.n_psu), int(dw.n_str)

    print("E0 group table / dropout")
    group_table(df, cfg).to_csv(out("E0_group_sizes.csv"), index=False)
    if args.full:
        full = pd.read_csv(args.full, low_memory=False)
        dd, miss, n_drop, n_keep = dropout_check(full, cfg)
        dd.to_csv(out("E0_dropout_check.csv"), index=False)
        miss.to_csv(out("E0_missing_share.csv"), index=False)
        summary.update(n_dropped=n_drop, n_kept=n_keep)

    print("E7 Cramer's V")
    cramers_table(df, cfg, dw, du).to_csv(out("E7_cramers_v.csv"), index=False)

    print("E1 mining, unweighted and weighted")
    ru = mine(df, cfg, weighted=False)
    rw = mine(df, cfg, weighted=True, design=dw)
    summary["n_candidates_generated"] = ru.attrs["n_candidates_generated"]
    summary["n_candidates_after_min_count"] = {"1-item": int((ru["length"] == 1).sum()),
                                               "2-item": int((ru["length"] == 2).sum())}
    summary["n_significant_unweighted"] = int(ru["significant"].sum())
    summary["n_significant_weighted"] = int(rw["significant"].sum())
    summary["n_holm_only_unweighted"] = int((ru["p_holm"] < cfg["alpha"]).sum())
    ru.drop(columns="pattern").to_csv(out("E1_candidates_unweighted.csv"), index=False)
    rw.drop(columns="pattern").to_csv(out("E1_candidates_weighted.csv"), index=False)

    print("E3 pruning")
    pu = prune(ru, df, cfg, weighted=False)
    pw = prune(rw, df, cfg, weighted=True)
    pu.drop(columns="pattern").to_csv(out("E3_pruning_unweighted.csv"), index=False)
    pw.drop(columns="pattern").to_csv(out("E3_pruning_weighted.csv"), index=False)
    final_u = pu[pu["kept"]].sort_values("delta", ascending=False)
    final_w = pw[pw["kept"]].sort_values("delta", ascending=False)
    summary["n_final_unweighted"] = int(len(final_u))
    summary["n_final_weighted"] = int(len(final_w))
    summary["n_final_pairs_weighted"] = int((final_w["length"] == 2).sum())
    summary["pruning_reasons_weighted"] = pw["drop_reason"].value_counts().to_dict()
    # agreement between weighted and unweighted results
    ku = set(final_u["label"]); kw = set(final_w["label"])
    summary["final_overlap_jaccard"] = len(ku & kw) / max(len(ku | kw), 1)
    mg = final_u.set_index("label")["max_group"]
    mgw = rw.set_index("label")["max_group"]
    summary["same_max_group_weighted_vs_unweighted"] = int(sum(mgw.get(l) == g for l, g in mg.items()))

    print("E2 one-vs-rest")
    ovr = pd.concat([one_vs_rest(df, cfg, p, design=dw) for p in final_w["pattern"]], ignore_index=True)
    ovr.to_csv(out("E2_one_vs_rest_weighted.csv"), index=False)
    best = ovr.loc[ovr.groupby("pattern")["diff"].apply(lambda s: s.abs().idxmax())]
    best.to_csv(out("E2_best_group_per_pattern.csv"), index=False)
    # how often the max-support group differs from the one-vs-rest group
    lab2max = final_w.set_index("label")["max_group"]
    lab2min = final_w.set_index("label")["min_group"]
    summary["ovr_group_equals_max_group"] = int(sum(lab2max[r.pattern] == r.group for r in best.itertuples()))
    summary["ovr_group_equals_min_group"] = int(sum(lab2min[r.pattern] == r.group for r in best.itertuples()))

    print("E4 sensitivity")
    rows = []
    base = set(final_w["label"])
    for dlt in cfg["sens_delta"]:
        for mc in cfg["sens_min_count"]:
            r = mine(df, cfg, weighted=True, design=dw, delta=dlt, min_count=mc)
            p = prune(r, df, cfg, weighted=True)
            k = p[p["kept"]]
            labs = set(k["label"])
            rows.append(dict(delta=dlt, min_count=mc, n_significant=int(r["significant"].sum()),
                             n_final=len(k), n_final_pairs=int((k["length"] == 2).sum()),
                             jaccard_vs_base=len(labs & base) / max(len(labs | base), 1)))
    pd.DataFrame(rows).to_csv(out("E4_sensitivity.csv"), index=False)

    print(f"E5 bootstrap (B={args.boot})")
    bs = bootstrap_stability(df, cfg, final_w, weighted=True, B=args.boot, rng=rng)
    bs.to_csv(out("E5_bootstrap_stability.csv"), index=False)

    print("E6 baselines")
    mn, r2 = baseline_multinomial(df, cfg)
    mn.to_csv(out("E6_multinomial_logit.csv"), index=False)
    summary["multinomial_mcfadden_r2"] = r2
    wr = baseline_wracc(df, cfg, weighted=True)
    wr.to_csv(out("E6_wracc_top10.csv"), index=False)
    summary["wracc_top10_pairs"] = int((wr["length"] == 2).sum())
    summary["wracc_top_overlap_with_final"] = int(wr["pattern"].isin(final_w["label"]).sum())

    print("E8 circularity")
    present = {k: v for k, v in cfg["conditions"].items() if v in df}
    if present:
        rows = []
        for name, col in present.items():
            for gr in cfg["groups"]:
                k = df[cfg["group"]] == gr
                rows.append(dict(condition=name, group=gr,
                                 pct=100 * df.loc[k, col].mean(),
                                 pct_weighted=100 * np.average(df.loc[k, col], weights=df.loc[k, cfg["weight"]])))
        pd.DataFrame(rows).pivot(index="condition", columns="group", values="pct_weighted") \
          .reindex(columns=cfg["groups"]).to_csv(out("E8_conditions_by_group.csv"))
    no_out = [a for a in cfg["attrs"] if a not in cfg["outcome_attrs"]]
    r8 = mine(df, cfg, attrs=no_out, weighted=True, design=dw)
    p8 = prune(r8, df, cfg, weighted=True)
    p8[p8["kept"]].drop(columns="pattern").to_csv(out("E8_final_without_outcome_attrs.csv"), index=False)
    summary["n_final_without_outcome_attrs"] = int(p8["kept"].sum())

    print("E9 age-stratified (same pipeline, age removed)")
    no_age = [a for a in cfg["attrs"] if a != cfg["age_attr"]]
    u65 = df[df[cfg["age_attr"]].isin(cfg["under65_codes"])].reset_index(drop=True)
    o65 = df[~df[cfg["age_attr"]].isin(cfg["under65_codes"])].reset_index(drop=True)
    res_age = {}
    for name, sub in [("under65", u65), ("65plus", o65)]:
        dsub = Design(sub, cfg, weighted=True)
        r = mine(sub, cfg, attrs=no_age, weighted=True, design=dsub)
        p = prune(r, sub, cfg, weighted=True)
        k = p[p["kept"]]
        k.drop(columns="pattern").to_csv(out(f"E9_final_{name}.csv"), index=False)
        summary[f"age_{name}_n"] = len(sub)
        summary[f"age_{name}_significant_before_pruning"] = int(r["significant"].sum())
        summary[f"age_{name}_final_after_pruning"] = int(len(k))
        res_age[name] = r[r["significant"]]
    fig_age(res_age["under65"], res_age["65plus"], out("fig_age_comparison.png"))

    print("E10 single-condition groups only")
    r10 = mine(df, cfg, groups=cfg["single_condition_groups"], weighted=True)
    p10 = prune(r10, df, cfg, groups=cfg["single_condition_groups"], weighted=True)
    p10[p10["kept"]].drop(columns="pattern").to_csv(out("E10_single_condition_groups.csv"), index=False)
    summary["n_final_single_condition_groups"] = int(p10["kept"].sum())

    print("Synergy recount (explains 793 vs 828)")
    pairs = ru[ru["length"] == 2]
    summary["pairs_generated"] = int(sum(1 for c in candidates(build_items(df, cfg["attrs"]), 2) if len(c) == 2))
    summary["pairs_after_min_count"] = int(len(pairs))

    print("Figures")
    fig_heatmap(final_w, cfg, out("fig_heatmap_weighted.png"))
    fig_forest(ovr, out("fig_forest_one_vs_rest.png"))
    final_w.drop(columns="pattern").to_csv(out("final_patterns_weighted.csv"), index=False)

    with open(out("summary.json"), "w") as f:
        json.dump(summary, f, indent=2, default=float)
    print(json.dumps(summary, indent=2, default=float))


if __name__ == "__main__":
    main()


### 1.3 `fast_rs.py`: fast version of the same design-based test (used for the bootstrap)
`boot7.py` checks that this gives exactly the same decisions as the main pipeline before it starts.

In [ ]:
%%writefile fast_rs.py
"""Vectorized design-based test for many binary patterns at once.

For each candidate pattern (a 0/1 column), tests H0: equal weighted support in all
K groups. Uses the same statistic as csm_upgrade.wald_homogeneity with J = 1:
Taylor-linearized covariance of the K domain proportions under a stratified
with-replacement PSU design, and the first-order Rao-Scott correction
RS1 = X2_srs / mean_deff, referred to chi2 with K-1 df.
Used for the main analysis check and for every bootstrap replicate (M2)."""
import numpy as np
from scipy import stats


def design_arrays(strata, psu):
    """Return PSU index per row, stratum index per PSU, and n_h."""
    import pandas as pd
    key = pd.Series(strata).astype(str) + "|" + pd.Series(psu).astype(str)
    psu_idx, keys = pd.factorize(key)
    str_of_psu, _ = pd.factorize(pd.Series(keys).str.split("|").str[0])
    nh = np.bincount(str_of_psu)
    return psu_idx, str_of_psu, nh


def rs1_batch(Y, gidx, K, w, psu_idx, str_of_psu, nh, chunk=64):
    """Y: n x C (0/1). gidx: group code 0..K-1 per row. Returns supports (C x K),
    RS1 statistic, p-value and mean design effect per candidate."""
    n, C = Y.shape
    n_psu, H = len(str_of_psu), len(nh)
    f = np.where(nh > 1, nh / np.maximum(nh - 1, 1), 0.0)
    sqf = np.sqrt(f)[str_of_psu]                      # per PSU
    Wk = np.bincount(gidx, weights=w, minlength=K)
    nk = np.bincount(gidx, minlength=K)
    q = K - 1
    Cm = np.zeros((q, K)); Cm[np.arange(q), np.arange(q)] = 1; Cm[:, K - 1] = -1
    P_all = np.zeros((C, K)); rs1 = np.zeros(C); deff = np.zeros(C)
    for s in range(0, C, chunk):
        Yc = Y[:, s:s + chunk].astype(float); c = Yc.shape[1]
        P = np.zeros((K, c)); D = []
        for k in range(K):
            m = gidx == k
            P[k] = (w[m] @ Yc[m]) / Wk[k]
            U = np.zeros((n, c))
            U[m] = (w[m, None] * (Yc[m] - P[k])) / Wk[k]
            T = np.zeros((n_psu, c)); np.add.at(T, psu_idx, U)       # PSU totals
            S = np.zeros((H, c)); np.add.at(S, str_of_psu, T)
            T -= (S / nh[:, None])[str_of_psu]                         # centre in stratum
            D.append(T * sqf[:, None])
        V = np.zeros((c, K, K))
        for k in range(K):
            for l in range(k, K):
                v = np.einsum("ij,ij->j", D[k], D[l]); V[:, k, l] = v; V[:, l, k] = v
        Vs = np.zeros((c, K, K))
        for k in range(K):
            Vs[:, k, k] = P[k] * (1 - P[k]) / nk[k]
        cp = (Cm @ P).T                                                # c x q
        CVC = Cm @ V @ Cm.T
        CVsC = Cm @ Vs @ Cm.T
        iS = np.linalg.pinv(CVsC)
        Ws = np.einsum("ci,cij,cj->c", cp, iS, cp)
        dm = np.trace(iS @ CVC, axis1=1, axis2=2) / q
        P_all[s:s + c] = P.T; rs1[s:s + c] = Ws / np.maximum(dm, 1e-12); deff[s:s + c] = dm
    return P_all, rs1, stats.chi2.sf(rs1, q), deff


### 1.4 `make_7groups.py`: splits the old 'other single condition' group into Diab.+depression and Diab.+COPD

In [ ]:
%%writefile make_7groups.py
"""Task A: split 'diabetes_other_single' into diabetes_depression and diabetes_copd."""
import pandas as pd
for src,dst in [("analysis_corrected.csv","analysis_7g.csv"),("diabetes_all_corrected.csv","diabetes_all_7g.csv")]:
    d=pd.read_csv(src,low_memory=False)
    os_=d.mm_group=="diabetes_other_single"
    assert ((d.cond_depression+d.cond_copd)[os_]==1).all()
    d.loc[os_&(d.cond_depression==1),"mm_group"]="diabetes_depression"
    d.loc[os_&(d.cond_copd==1),"mm_group"]="diabetes_copd"
    d.to_csv(dst,index=False); print(dst,d.mm_group.value_counts().to_dict())


### 1.5 `run7.py`: runs the main pipeline with the seven groups
(`--boot 2` here only makes a 2-sample placeholder file, `E5_bootstrap_stability.csv`. It is **not** used. The real bootstrap is `boot7.py`.)

In [ ]:
%%writefile run7.py
import sys; sys.path.insert(0,'.')
import csm_upgrade as C
G7=["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G7,single_condition_groups=["diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd"])
C.GROUP_NAMES.update({"diabetes_depression":"Diab.+depression","diabetes_copd":"Diab.+COPD"})
sys.argv=["csm_upgrade.py","--data","analysis_7g.csv","--full","diabetes_all_7g.csv","--out","results7","--boot","2"]
C.main()


### 1.6 `boot7.py`: the 200-sample design-based bootstrap

In [ ]:
%%writefile boot7.py
"""200-sample bootstrap, design-based Rao-Scott test (same as main analysis), seven groups.
Respondents are resampled within strata (each respondent is its own PSU in BRFSS 2024)."""
import sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, time, json
import csm_upgrade as C, fast_rs as F
G=["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg=C.CFG; K=len(G); B=int(sys.argv[1])
df=pd.read_csv("analysis_7g.csv",low_memory=False)
for a in cfg["attrs"]: df[a]=df[a].map(C.norm_code)
g=pd.Categorical(df.mm_group,categories=G).codes
cands=C.candidates(C.build_items(df,cfg["attrs"]),2); cache={}
M=np.column_stack([C.mask_of(df,c,cache) for c in cands]); keep=M.sum(0)>=30; M=M[:,keep]
lab=[C.pattern_label(c) for c,k in zip(cands,keep) if k]
w=df._LLCPWT.values.astype(float); st=df._STSTR.values
# check full-data decisions equal the main run
pi,sp,nh=F.design_arrays(st,df._PSU.values)
P,rs,p,de=F.rs1_batch(M,g,K,w,pi,sp,nh); sig=(C.holm(p)<.05)&(P.max(1)-P.min(1)>=.1)
e1=pd.read_csv("results7/E1_candidates_weighted.csv"); assert set(np.array(lab)[sig])==set(e1[e1.significant].label), "mismatch"
final=pd.read_csv("results7/final_patterns_weighted.csv"); fl=final.label.tolist(); fmax=dict(zip(final.label,final.max_group))
pos={l:i for i,l in enumerate(lab)}; fidx=[pos[l] for l in fl]
sampler=C.PSUBootstrap(df,cfg,True); rng=np.random.default_rng(2026)
found=np.zeros(len(fl)); same=np.zeros(len(fl)); t0=time.time()
for b in range(B):
    idx=sampler.draw(rng); Mb=M[idx]; sp,_=pd.factorize(st[idx]); nh=np.bincount(sp)
    ok=np.where(Mb.sum(0)>=30)[0]
    P,rs,p,de=F.rs1_batch(Mb[:,ok],g[idx],K,w[idx],np.arange(len(idx)),sp,nh)
    s=(C.holm(p)<.05)&(P.max(1)-P.min(1)>=.1)
    smap={ok[j]:G[int(P[j].argmax())] for j in np.where(s)[0]}
    for t,(l,i) in enumerate(zip(fl,fidx)):
        if i in smap: found[t]+=1; same[t]+=(smap[i]==fmax[l])
    if b%25==0: print(b,round(time.time()-t0),flush=True)
pd.DataFrame(dict(pattern=fl,rediscovery_rate=found/B,same_max_group_rate=same/B)).to_csv("results7/E5_bootstrap_stability_raoscott.csv",index=False)
print("done",time.time()-t0)


### 1.7 `boot_compare7.py`: unweighted Pearson test vs design-based test on 20 bootstrap samples

In [ ]:
%%writefile boot_compare7.py
"""Diagnostic for M2: on the same bootstrap samples, compare decisions of the
unweighted Pearson test (old bootstrap) and the design-based Rao-Scott test (new)."""
import sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, json
from scipy import stats
import csm_upgrade as C, fast_rs as F
cfg=C.CFG; B=int(sys.argv[1])
df=pd.read_csv("analysis_7g.csv",low_memory=False)
for a in cfg["attrs"]: df[a]=df[a].map(C.norm_code)
G=["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]; g=pd.Categorical(df.mm_group,categories=G).codes
cands=C.candidates(C.build_items(df,cfg["attrs"]),2); cache={}
M=np.column_stack([C.mask_of(df,c,cache) for c in cands]); kp=M.sum(0)>=30; M=M[:,kp]; lab=[C.pattern_label(c) for c,k in zip(cands,kp) if k]
w=df._LLCPWT.values.astype(float); st=df._STSTR.values
fl=set(pd.read_csv("results7/final_patterns_weighted.csv").label); fmask=np.array([l in fl for l in lab])
sampler=C.PSUBootstrap(df,cfg,True); rng=np.random.default_rng(2026)
res=[]
for b in range(B):
    idx=sampler.draw(rng); Mb=M[idx]; gb=g[idx]; wb=w[idx]
    sp,_=pd.factorize(st[idx]); nh=np.bincount(sp); ok=np.where(Mb.sum(0)>=30)[0]
    P,rs,p_rs,de=F.rs1_batch(Mb[:,ok],gb,7,wb,np.arange(len(idx)),sp,nh)
    # unweighted Pearson on counts
    nk=np.bincount(gb,minlength=7); X=np.zeros(len(ok))
    for j,c in enumerate(ok):
        x=np.bincount(gb[Mb[:,c]],minlength=7); tab=np.vstack([x,nk-x])
        X[j]=stats.chi2_contingency(tab,correction=False)[1] if (tab.sum(1)>0).all() else 1.0
    d=P.max(1)-P.min(1)
    s_rs=(C.holm(p_rs)<.05)&(d>=.1); s_pe=(C.holm(X)<.05)&(d>=.1)
    fin=fmask[ok]
    res.append(dict(rs=int(s_rs.sum()),pe=int(s_pe.sum()),pe_not_rs=int((s_pe&~s_rs).sum()),rs_not_pe=int((s_rs&~s_pe).sum()),
                    final_flips=int(((s_pe!=s_rs)&fin).sum()),delta_pass=int((d>=.1).sum()),
                    holm_rs=int((C.holm(p_rs)<.05).sum()),holm_pe=int((C.holm(X)<.05).sum())))
r=pd.DataFrame(res); print(r.describe().round(2)); json.dump(r.sum().to_dict()|{"B":B},open("results7/E5_bootstrap_test_comparison.json","w"),indent=1,default=int)
print(r.sum())


### 1.8 `gen7.py`: tables and the two figures

In [ ]:
%%writefile gen7.py
"""Tables and figures for the seven-group paper. All values read from results7/."""
import pandas as pd, numpy as np, json, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":7.5})
R="results7/"
G=["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
GS={"diabetes_only":"Only","diabetes_cvd":"CVD","diabetes_ckd":"CKD","diabetes_arthritis":"Arth.","diabetes_depression":"Dep.","diabetes_copd":"COPD","diabetes_multi_2plus":"2+"}
GH={"diabetes_only":"Only","diabetes_cvd":"CVD","diabetes_ckd":"CKD","diabetes_arthritis":"Arth.","diabetes_depression":"Dep.","diabetes_copd":"COPD","diabetes_multi_2plus":"2+"}
GN={"diabetes_only":"Diabetes only","diabetes_cvd":"Diab.+CVD","diabetes_ckd":"Diab.+CKD","diabetes_arthritis":"Diab.+arthritis","diabetes_depression":"Diab.+depression","diabetes_copd":"Diab.+COPD","diabetes_multi_2plus":"Diab.+2 or more"}
SH={"Poor mental health days: ":"MH days ","General health: ":"Health ","Physical activity: Any leisure activity":"Active",
    "Physical activity: No leisure activity":"Inactive","Smoking: Never smoked":"Never smoked","Smoking: Former smoker":"Former smoker",
    "Smoking: Current, daily":"Daily smoker","Education: College graduate":"College grad.","Income: ":"Income ","BMI: ":"","Age: ":"Age ",
    " AND ":" + ","0 days":"0","14-30 days":"14-30","1-5 days":"1-5","6-13 days":"6-13","Very good":"very good","Good":"good","Fair":"fair","Poor":"poor"}
def short(x,tex=False):
    for a,b in SH.items(): x=x.replace(a,b)
    return x.replace("$","\\$") if tex else x
T={}
f=pd.read_csv(R+"final_patterns_weighted.csv")
b=pd.read_csv(R+"E2_best_group_per_pattern.csv").set_index("pattern")
bs=pd.read_csv(R+"E5_bootstrap_stability_raoscott.csv").set_index("pattern")
# Table V: 12 largest |D|
b["absD"]=b["diff"].abs(); top=b.sort_values("absD",ascending=False).head(12); fi=f.set_index("label")
T["patterns"]="\n".join(f"{short(l,True)} & {fi.loc[l,'delta']:.3f} & {GS[fi.loc[l,'max_group']]} & {GS[fi.loc[l,'min_group']]} & {GS[x.group]} & ${x['diff']:+.3f}$ $[{x.diff_lo:+.3f},{x.diff_hi:+.3f}]$ & {x.rr:.2f} & {bs.loc[l,'rediscovery_rate']:.2f} & {bs.loc[l,'same_max_group_rate']:.2f}\\\\" for l,x in top.iterrows())
T["top12_min_absD"]=float(top.absD.min()); T["13th_absD"]=float(b.sort_values("absD",ascending=False).absD.iloc[12])
c=pd.read_csv(R+"E7_cramers_v.csv").replace({"Poor mental health days":"Poor MH days"})
T["cramer"]="\n".join(f"{r.attribute} & {r.levels} & {r.df_star} & {r.V_unweighted:.3f} & {r.V_weighted:.3f} & {r.label_weighted} & {r.mean_deff:.1f}\\\\" for r in c.itertuples())
e8=pd.read_csv(R+"E8_conditions_by_group.csv").set_index("condition")
order=[("CVD (heart attack, CHD or stroke)","CVD"),("Kidney disease","Kidney disease"),("Arthritis","Arthritis"),("Depressive disorder","Depressive disorder"),("COPD","COPD"),("Asthma (not used for groups)","Asthma$^\\ast$"),("Cancer (not used for groups)","Cancer$^\\ast$")]
T["comp"]="\n".join(n+" & "+" & ".join(f"{e8.loc[k,g]:.0f}" for g in G)+"\\\\" for k,n in order)
g=pd.read_csv(R+"E0_group_sizes.csv").set_index("mm_group"); allg=pd.read_csv("diabetes_all_7g.csv",usecols=["mm_group"]).mm_group.value_counts()
T["groups"]="\n".join(f"{GN[k]} & {allg[k]:,} & {int(g.loc[k,'n']):,} & {g.loc[k,'pct_weighted']:.1f}\\\\" for k in G)
json.dump(T,open("tables7.json","w"),indent=1)
# Fig 1 heatmap
fs=f.sort_values(["length","delta"],ascending=[True,False]); S=fs[["supp_"+x for x in G]].to_numpy()
fig,ax=plt.subplots(figsize=(3.45,5.3))
im=ax.imshow(S,cmap="Greys",vmin=0,vmax=1,aspect="auto")
for i in range(S.shape[0]):
    hi,lo=S[i].argmax(),S[i].argmin()
    for j in range(S.shape[1]):
        ax.text(j,i,f"{S[i,j]:.2f}"[1:] if S[i,j]<1 else "1.0",ha="center",va="center",fontsize=7,color="white" if S[i,j]>.55 else "black",fontweight="bold" if j in(hi,lo) else "normal")
    ax.add_patch(plt.Rectangle((hi-.5,i-.5),1,1,fill=False,lw=1.1,ec="black"))
    ax.add_patch(plt.Rectangle((lo-.5,i-.5),1,1,fill=False,lw=0.9,ec="black",ls=(0,(2,1))))
ax.axhline((fs.length==1).sum()-.5,color="black",lw=1.4)
ax.set_yticks(range(len(fs))); ax.set_yticklabels([short(l) for l in fs.label],fontsize=7)
ax.set_xticks(range(7)); ax.set_xticklabels([GH[x] for x in G],fontsize=7); ax.xaxis.tick_top(); ax.tick_params(length=0)
for s_ in ax.spines.values(): s_.set_visible(False)
fig.tight_layout(pad=0.3); fig.savefig("fig_heatmap_weighted.png",dpi=400); plt.close(fig)
# Fig 2 forest
bb=b.sort_values("diff"); fig,ax=plt.subplots(figsize=(3.45,4.9)); y=np.arange(len(bb))
ax.errorbar(bb["diff"],y,xerr=[bb["diff"]-bb.diff_lo,bb.diff_hi-bb["diff"]],fmt="o",color="black",ms=2.5,capsize=1.5,lw=0.8)
ax.axvline(0,color="0.5",lw=0.7,ls="--")
ax.set_yticks(y); ax.set_yticklabels([f"{short(p)} [{GS[q]}]" for p,q in zip(bb.index,bb.group)],fontsize=7)
ax.set_xlabel("Group minus rest (95% CI)",fontsize=7); ax.tick_params(axis="x",labelsize=7)
fig.tight_layout(pad=0.3); fig.savefig("fig_forest_one_vs_rest.png",dpi=400); plt.close(fig)
print(json.dumps({k:v for k,v in T.items() if not isinstance(v,str)}))


### 1.9 `counts7.py`, `compare6v7.py`, `runtime7.py`: small helper scripts
These produce `counts_7groups.json`, `compare_6_vs_7_groups.json` and `runtime.json`.

In [ ]:
%%writefile counts7.py
"""Counts quoted in the text: Holm passes, Delta passes, and patterns that fail only the design-based test."""
import json, pandas as pd
K = {"adults_with_diabetes": int(len(pd.read_csv("diabetes_all_7g.csv", usecols=["mm_group"])))}
K["group_sizes_all_diabetes"] = pd.read_csv("diabetes_all_7g.csv", usecols=["mm_group"]).mm_group.value_counts().to_dict()
for n in ["weighted", "unweighted"]:
    e = pd.read_csv(f"results7/E1_candidates_{n}.csv"); h = e.p_holm < .05; d = e.delta >= .1
    K[f"{n}_holm_pass"] = int(h.sum()); K[f"{n}_delta_pass"] = int(d.sum())
    K[f"{n}_delta_pass_fail_holm"] = int((d & ~h).sum())
json.dump(K, open("results7/counts_7groups.json", "w"), indent=1); print(json.dumps(K, indent=1))


In [ ]:
%%writefile compare6v7.py
"""Six-group vs seven-group final patterns."""
import json, pandas as pd
a = set(pd.read_csv("results6/final_patterns_weighted.csv").label)
b = set(pd.read_csv("results7/final_patterns_weighted.csv").label)
R = dict(six_group_final=len(a), seven_group_final=len(b), kept=len(a & b), new=sorted(b - a),
         dropped=sorted(a - b), jaccard=len(a & b) / len(a | b))
json.dump(R, open("results7/compare_6_vs_7_groups.json", "w"), indent=1); print(json.dumps(R, indent=1))


In [ ]:
%%writefile runtime7.py
"""Times the candidate build and the 814 design-based tests on the full data."""
import sys, time, json; sys.path.insert(0, ".")
import numpy as np, pandas as pd
import csm_upgrade as C, fast_rs as F
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg = C.CFG
df = pd.read_csv("analysis_7g.csv", low_memory=False)
for a in cfg["attrs"]: df[a] = df[a].map(C.norm_code)
g = pd.Categorical(df.mm_group, categories=G).codes
t0 = time.time(); cands = C.candidates(C.build_items(df, cfg["attrs"]), 2); cache = {}
M = np.column_stack([C.mask_of(df, c, cache) for c in cands]); M = M[:, M.sum(0) >= 30]; t1 = time.time()
pi, sp, nh = F.design_arrays(df._STSTR.values, df._PSU.values)
F.rs1_batch(M, g, len(G), df._LLCPWT.values.astype(float), pi, sp, nh); t2 = time.time()
R = {"n_tests": int(M.shape[1]), "candidate_build_s": round(t1 - t0, 1), "design_tests_s": round(t2 - t1, 1)}
json.dump(R, open("results7/runtime_this_pc.json", "w"), indent=1); print(R)


### 1.10 `legacy_check.py`: the missing-code check in the Data section

In [ ]:
%%writefile legacy_check.py
"""Missing-code check (Data section): keep BRFSS "don't know/refused" codes for smoking and age as
their own categories (legacy file) and compare the gap of the six-group single traits with the
corrected file. Same logic as the original side analysis. Unweighted, six groups."""
import sys, json; sys.path.insert(0, ".")
import pandas as pd
import csm_upgrade as C
cfg = C.CFG                                   # default config = the six groups
L = pd.read_csv("analysis_legacy.csv", low_memory=False); K = pd.read_csv("analysis_corrected.csv", low_memory=False)
for df in (L, K):
    for x in cfg["attrs"]: df[x] = df[x].map(C.norm_code)
rl = C.mine(L, cfg); rk = C.mine(K, cfg)
fl = rl[(rl.length == 1) & rl.significant][["label", "delta", "max_group"]]
cmp = fl.join(rk.set_index("label")[["delta", "max_group", "significant"]], on="label", rsuffix="_corr")
cmp.to_csv("results7/legacy_vs_corrected.csv", index=False)
R = {"n_single_traits": len(cmp), "max_abs_gap_change": float((cmp.delta - cmp.delta_corr).abs().max())}
json.dump(R, open("results7/legacy_vs_corrected.json", "w"), indent=1); print(cmp.round(4).to_string()); print(R)


## 2. Build the data
Expected output: `corrected diabetes: 65809 analysis: 47873`.

In [ ]:
run_script('build_data.py', RAW_CSV)
import pandas as pd
RAW_ROWS = len(pd.read_csv(RAW_CSV, usecols=['DIABETE4']))
print('raw rows:', RAW_ROWS, '(paper: 457,670)')

In [ ]:
run_script('make_7groups.py')

## 3. Main analysis (seven groups)

In [ ]:
T = {}
T['main_pipeline_s'] = run_script('run7.py')
run_script('counts7.py')

### Missing-code check (paper: gap changes by at most 0.008)

In [ ]:
run_script('legacy_check.py')

## 4. Six groups vs seven groups (Section V-E of the paper)

In [ ]:
if RUN_SIX_GROUP_COMPARISON:
    run_script('csm_upgrade.py', '--data', 'analysis_corrected.csv', '--full', 'diabetes_all_corrected.csv', '--out', 'results6', '--boot', '2')
    run_script('compare6v7.py')

## 5. Run time of the design-based tests

In [ ]:
run_script('runtime7.py')

## 6. Pearson vs design-based test on 20 bootstrap samples

In [ ]:
run_script('boot_compare7.py', 20)

## 7. Design-based bootstrap (slow: about 34 minutes for 200 samples on one core)

In [ ]:
T['bootstrap_s'] = run_script('boot7.py', BOOT_B)
print('bootstrap per sample (s):', round(T['bootstrap_s'] / BOOT_B, 1))

## 8. Tables and figures

In [ ]:
run_script('gen7.py')
from IPython.display import Image, display
display(Image('fig_heatmap_weighted.png', width=420)); display(Image('fig_forest_one_vs_rest.png', width=420))

## 9. Verify: your numbers vs the numbers in the paper
Each row reads your result files and compares them with the value I got. Counts must match exactly. Rates and ratios must match to 3 decimals (tolerance 0.002).

In [ ]:
import json, numpy as np, pandas as pd
def key_numbers(R="results7/", ALL="diabetes_all_7g.csv"):
    K={}; s=json.load(open(R+"summary.json"))
    for k in ["n_analysis","n_dropped","n_strata","n_significant_weighted","n_significant_unweighted","n_final_weighted",
              "n_final_pairs_weighted","n_final_unweighted","same_max_group_weighted_vs_unweighted","ovr_group_equals_max_group",
              "ovr_group_equals_min_group","wracc_top_overlap_with_final","n_final_without_outcome_attrs",
              "age_under65_final_after_pruning","age_65plus_final_after_pruning"]: K[k]=s[k]
    K["n_candidates_1item"]=s["n_candidates_after_min_count"]["1-item"]; K["n_candidates_2item"]=s["n_candidates_after_min_count"]["2-item"]
    K["R2_removed"]=s["pruning_reasons_weighted"].get("R2 not surprising given parents",0)
    K["R1_removed"]=s["pruning_reasons_weighted"].get("R1 complement of a kept item",0)
    K["multinomial_mcfadden_r2"]=round(s["multinomial_mcfadden_r2"],3)
    a=pd.read_csv(ALL,usecols=["mm_group"]).mm_group.value_counts(); K["adults_with_diabetes"]=int(a.sum())
    g=pd.read_csv(R+"E0_group_sizes.csv").set_index("mm_group")
    for x in g.index: K[f"n_all_{x}"]=int(a[x]); K[f"n_analysis_{x}"]=int(g.loc[x,"n"]); K[f"pct_weighted_{x}"]=round(float(g.loc[x,"pct_weighted"]),1)
    for n in ["weighted","unweighted"]:
        e=pd.read_csv(R+f"E1_candidates_{n}.csv"); h=e.p_holm<.05; d=e.delta>=.1
        K[f"{n}_holm_pass"]=int(h.sum()); K[f"{n}_delta_pass"]=int(d.sum()); K[f"{n}_delta_pass_fail_holm"]=int((d&~h).sum())
    b=pd.read_csv(R+"E2_best_group_per_pattern.csv")
    K["ci_excludes_0"]=int(((b.diff_lo>0)|(b.diff_hi<0)).sum())
    bs=pd.read_csv(R+"E5_bootstrap_stability_raoscott.csv")
    K["boot_found_all"]=int((bs.rediscovery_rate==1).sum()); K["boot_ge_0.9"]=int((bs.rediscovery_rate>=.9).sum())
    K["boot_ge_0.7"]=int((bs.rediscovery_rate>=.7).sum()); K["boot_min"]=round(float(bs.rediscovery_rate.min()),3)
    K["boot_same_group_ge_0.9"]=int((bs.same_max_group_rate>=.9).sum())
    o=pd.read_csv(R+"E2_one_vs_rest_weighted.csv").set_index(["pattern","group"])
    def ov(p,gr,cols):
        r=o.loc[(p,gr)]; return {c:round(float(r[c]),3) for c in cols}
    for k,v in ov("Smoking: Never smoked","diabetes_copd",["supp_group","supp_rest","diff"]).items(): K["copd_never_smoked_"+k]=v
    for k,v in ov("Smoking: Current, daily","diabetes_copd",["rr","rr_lo","rr_hi"]).items(): K["copd_daily_smoker_"+k]=v
    for k,v in ov("Poor mental health days: 0 days","diabetes_depression",["supp_group","supp_rest","diff","rr"]).items(): K["dep_mh0_"+k]=v
    for k,v in ov("General health: Poor","diabetes_multi_2plus",["supp_group","supp_rest","rr"]).items(): K["multi_poor_health_"+k]=v
    c=pd.read_csv(R+"E7_cramers_v.csv"); K["cramer_V_weighted_max"]=round(float(c.V_weighted.max()),3)
    K["deff_min"]=round(float(c.mean_deff.min()),1); K["deff_max"]=round(float(c.mean_deff.max()),1)
    se=pd.read_csv(R+"E4_sensitivity.csv")
    for r in se.itertuples(): K[f"sens_d{r.delta}_m{r.min_count}_final"]=int(r.n_final); K[f"sens_d{r.delta}_m{r.min_count}_jaccard"]=round(float(r.jaccard_vs_base),2)
    try:
        t=json.load(open(R+"E5_bootstrap_test_comparison.json")); K["pearson_extra_passes_20"]=int(t["pe_not_rs"]); K["final_pattern_flips_20"]=int(t["final_flips"])
    except FileNotFoundError: pass

    # ---- extra values quoted in the text ----
    K["n_removed_pct"]=round(100*s["n_dropped"]/(s["n_dropped"]+s["n_analysis"]),1)
    ms=pd.read_csv(R+"E0_missing_share.csv").set_index("index").share_missing; K["income_missing_pct"]=round(100*float(ms["income_group"]),1)
    dd=pd.read_csv(R+"E0_dropout_check.csv"); dd["level"]=dd.level.astype(str)
    fem=dd[(dd.variable=="_SEX")&(dd.level.isin(["2","2.0"]))].iloc[0]; a80=dd[(dd.variable=="age_group")&(dd.level.isin(["13","13.0"]))].iloc[0]
    K["removed_female_pct"]=round(float(fem.pct_dropped),1); K["kept_female_pct"]=round(float(fem.pct_kept),1)
    K["removed_age80_pct"]=round(float(a80.pct_dropped),1); K["kept_age80_pct"]=round(float(a80.pct_kept),1)
    try: K["missing_code_max_gap_change"]=round(json.load(open(R+"legacy_vs_corrected.json"))["max_abs_gap_change"],3)
    except FileNotFoundError: pass
    f=pd.read_csv(R+"final_patterns_weighted.csv"); K["final_single_values"]=int((f.length==1).sum())
    u=pd.read_csv(R+"E3_pruning_unweighted.csv"); K["unweighted_and_weighted_common"]=len(set(u[u.kept].label)&set(f.label))
    bb=b.set_index("pattern"); bsi=bs.set_index("pattern")
    inv=[l for l,r in f.set_index("label").iterrows() if "diabetes_copd" in (r.max_group,r.min_group) or bb.loc[l,"group"]=="diabetes_copd"]
    K["copd_patterns"]=len(inv); K["copd_patterns_min_rediscovery"]=round(float(bsi.loc[inv].rediscovery_rate.min()),2)
    K["copd_patterns_mean_rediscovery"]=round(float(bsi.loc[inv].rediscovery_rate.mean()),2)
    K["mh0_same_group_rate"]=round(float(bsi.loc["Poor mental health days: 0 days","same_max_group_rate"]),3)
    K["multi2plus_characteristic_count"]=int((b.group=="diabetes_multi_2plus").sum())
    K["multi_active_D"]=ov("Physical activity: Any leisure activity","diabetes_multi_2plus",["diff"])["diff"]
    K["multi_never_smoked_D"]=ov("Smoking: Never smoked","diabetes_multi_2plus",["diff"])["diff"]
    m=pd.read_csv(R+"E6_multinomial_logit.csv"); sg=f[f.length==1].label; sig=set(m[(m.lo>1)|(m.hi<1)].term)
    K["mn_singles_in_model"]=int(sg.isin(set(m.term)).sum()); K["mn_singles_OR_CI_excludes_1"]=int(sg.isin(sig).sum())
    w=pd.read_csv(R+"E6_wracc_top10.csv"); K["wracc_top10_total"]=len(w); K["wracc_top10_in_final"]=int(w.pattern.isin(f.label).sum())
    K["wracc_top1_is_mental_health"]=int(w[w["rank"]==1].pattern.str.startswith("Poor mental health").sum())
    for n in ["under65","65plus"]:
        e=pd.read_csv(R+f"E9_final_{n}.csv"); K[f"mh14_30_delta_{n}"]=round(float(e[e.label=="Poor mental health days: 14-30 days"].delta.iloc[0]),3)
    K["unweighted_final_pairs"]=int((u[u.kept].length==2).sum())
    for r in se.itertuples(): K[f"sens_d{r.delta}_m{r.min_count}_pairs"]=int(r.n_final_pairs)
    return K


EXPECTED = {
 "n_analysis": 47873,
 "n_dropped": 17936,
 "n_strata": 2034,
 "n_significant_weighted": 78,
 "n_significant_unweighted": 79,
 "n_final_weighted": 30,
 "n_final_pairs_weighted": 12,
 "n_final_unweighted": 31,
 "same_max_group_weighted_vs_unweighted": 25,
 "ovr_group_equals_max_group": 8,
 "ovr_group_equals_min_group": 20,
 "wracc_top_overlap_with_final": 31,
 "n_final_without_outcome_attrs": 13,
 "age_under65_final_after_pruning": 25,
 "age_65plus_final_after_pruning": 19,
 "n_candidates_1item": 45,
 "n_candidates_2item": 769,
 "R2_removed": 47,
 "R1_removed": 1,
 "multinomial_mcfadden_r2": 0.126,
 "adults_with_diabetes": 65809,
 "n_all_diabetes_only": 17147,
 "n_analysis_diabetes_only": 12346,
 "pct_weighted_diabetes_only": 30.8,
 "n_all_diabetes_cvd": 3561,
 "n_analysis_diabetes_cvd": 2537,
 "pct_weighted_diabetes_cvd": 5.0,
 "n_all_diabetes_ckd": 1347,
 "n_analysis_diabetes_ckd": 962,
 "pct_weighted_diabetes_ckd": 2.3,
 "n_all_diabetes_arthritis": 12942,
 "n_analysis_diabetes_arthritis": 9238,
 "pct_weighted_diabetes_arthritis": 18.1,
 "n_all_diabetes_depression": 3537,
 "n_analysis_diabetes_depression": 2676,
 "pct_weighted_diabetes_depression": 6.4,
 "n_all_diabetes_copd": 942,
 "n_analysis_diabetes_copd": 672,
 "pct_weighted_diabetes_copd": 1.3,
 "n_all_diabetes_multi_2plus": 26333,
 "n_analysis_diabetes_multi_2plus": 19442,
 "pct_weighted_diabetes_multi_2plus": 36.1,
 "weighted_holm_pass": 503,
 "weighted_delta_pass": 92,
 "weighted_delta_pass_fail_holm": 14,
 "unweighted_holm_pass": 614,
 "unweighted_delta_pass": 79,
 "unweighted_delta_pass_fail_holm": 0,
 "ci_excludes_0": 30,
 "boot_found_all": 17,
 "boot_ge_0.9": 23,
 "boot_ge_0.7": 27,
 "boot_min": 0.64,
 "boot_same_group_ge_0.9": 4,
 "copd_never_smoked_supp_group": 0.182,
 "copd_never_smoked_supp_rest": 0.547,
 "copd_never_smoked_diff": -0.365,
 "copd_daily_smoker_rr": 2.497,
 "copd_daily_smoker_rr_lo": 1.81,
 "copd_daily_smoker_rr_hi": 3.446,
 "dep_mh0_supp_group": 0.242,
 "dep_mh0_supp_rest": 0.61,
 "dep_mh0_diff": -0.369,
 "dep_mh0_rr": 0.396,
 "multi_poor_health_supp_group": 0.22,
 "multi_poor_health_supp_rest": 0.064,
 "multi_poor_health_rr": 3.447,
 "cramer_V_weighted_max": 0.224,
 "deff_min": 5.6,
 "deff_max": 7.1,
 "sens_d0.05_m30_final": 56,
 "sens_d0.05_m30_jaccard": 0.51,
 "sens_d0.05_m100_final": 56,
 "sens_d0.05_m100_jaccard": 0.51,
 "sens_d0.05_m500_final": 54,
 "sens_d0.05_m500_jaccard": 0.53,
 "sens_d0.1_m30_final": 30,
 "sens_d0.1_m30_jaccard": 1.0,
 "sens_d0.1_m100_final": 30,
 "sens_d0.1_m100_jaccard": 1.0,
 "sens_d0.1_m500_final": 30,
 "sens_d0.1_m500_jaccard": 1.0,
 "sens_d0.15_m30_final": 17,
 "sens_d0.15_m30_jaccard": 0.52,
 "sens_d0.15_m100_final": 17,
 "sens_d0.15_m100_jaccard": 0.52,
 "sens_d0.15_m500_final": 17,
 "sens_d0.15_m500_jaccard": 0.52,
 "pearson_extra_passes_20": 215,
 "final_pattern_flips_20": 2,
 "n_removed_pct": 27.3,
 "income_missing_pct": 18.7,
 "removed_female_pct": 58.5,
 "kept_female_pct": 46.9,
 "removed_age80_pct": 16.5,
 "kept_age80_pct": 11.4,
 "missing_code_max_gap_change": 0.008,
 "final_single_values": 18,
 "unweighted_and_weighted_common": 26,
 "copd_patterns": 9,
 "copd_patterns_min_rediscovery": 0.76,
 "copd_patterns_mean_rediscovery": 0.96,
 "mh0_same_group_rate": 0.445,
 "multi2plus_characteristic_count": 9,
 "multi_active_D": -0.141,
 "multi_never_smoked_D": -0.164,
 "mn_singles_in_model": 15,
 "mn_singles_OR_CI_excludes_1": 15,
 "wracc_top10_total": 70,
 "wracc_top10_in_final": 31,
 "wracc_top1_is_mental_health": 7,
 "mh14_30_delta_under65": 0.365,
 "mh14_30_delta_65plus": 0.202,
 "unweighted_final_pairs": 11,
 "sens_d0.05_m30_pairs": 25,
 "sens_d0.05_m100_pairs": 25,
 "sens_d0.05_m500_pairs": 25,
 "sens_d0.1_m30_pairs": 12,
 "sens_d0.1_m100_pairs": 12,
 "sens_d0.1_m500_pairs": 12,
 "sens_d0.15_m30_pairs": 6,
 "sens_d0.15_m100_pairs": 6,
 "sens_d0.15_m500_pairs": 6,
 "raw_records": 457670
}
EXPECTED_SIX_VS_SEVEN = {"six_group_final": 24, "kept": 20, "new": 10, "dropped": 4, "jaccard": 0.588}

mine = key_numbers("results7/", "diabetes_all_7g.csv"); mine["raw_records"] = RAW_ROWS
rows = []
for k, v in EXPECTED.items():
    m = mine.get(k)
    ok = (m is not None) and (abs(m - v) <= (0 if isinstance(v, int) else 0.002 + 1e-9))
    rows.append((k, v, m, "PASS" if ok else "FAIL"))
if os.path.exists("results7/compare_6_vs_7_groups.json"):
    c = json.load(open("results7/compare_6_vs_7_groups.json"))
    got = {"six_group_final": c["six_group_final"], "kept": c["kept"], "new": len(c["new"]),
           "dropped": len(c["dropped"]), "jaccard": round(c["jaccard"], 3)}
    for k, v in EXPECTED_SIX_VS_SEVEN.items():
        rows.append(("six_vs_seven_" + k, v, got[k], "PASS" if abs(got[k] - v) <= 0.001 else "FAIL"))
V = pd.DataFrame(rows, columns=["number", "paper (my run)", "your run", "result"])
pd.set_option("display.max_rows", 200); display(V)
print(f"PASS {(V.result=='PASS').sum()} of {len(V)}")
if BOOT_B != 200: print("Note: BOOT_B is not 200, so the boot_* rows are expected to differ.")

## 10. Every table and every number in the paper
`paper_all.py` rebuilds Tables I to VII and lists every number quoted in the text, all read from your result files. Fig. 1 and Fig. 2 are shown in Section 8 and saved as `fig_heatmap_weighted.png` and `fig_forest_one_vs_rest.png`.

In [ ]:
%%writefile paper_all.py
"""Rebuilds every table of the paper and every number quoted in its text, from the result files."""
import json, numpy as np, pandas as pd
R = "results7/"
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
GN = {"diabetes_only":"Diabetes only","diabetes_cvd":"Diab.+CVD","diabetes_ckd":"Diab.+CKD","diabetes_arthritis":"Diab.+arthritis",
      "diabetes_depression":"Diab.+depression","diabetes_copd":"Diab.+COPD","diabetes_multi_2plus":"Diab.+2 or more"}
GS = {"diabetes_only":"Only","diabetes_cvd":"CVD","diabetes_ckd":"CKD","diabetes_arthritis":"Arth.","diabetes_depression":"Dep.","diabetes_copd":"COPD","diabetes_multi_2plus":"2+"}
s = json.load(open(R+"summary.json")); cnt = json.load(open(R+"counts_7groups.json"))
f = pd.read_csv(R+"final_patterns_weighted.csv"); fi = f.set_index("label")
b = pd.read_csv(R+"E2_best_group_per_pattern.csv").set_index("pattern")
o = pd.read_csv(R+"E2_one_vs_rest_weighted.csv").set_index(["pattern","group"])
bs = pd.read_csv(R+"E5_bootstrap_stability_raoscott.csv").set_index("pattern")
ALL = pd.read_csv("diabetes_all_7g.csv", usecols=["mm_group"]).mm_group.value_counts()

def tables():
    T = {}
    g = pd.read_csv(R+"E0_group_sizes.csv").set_index("mm_group")
    t1 = pd.DataFrame({"Group":[GN[k] for k in G], "All":[int(ALL[k]) for k in G], "Analysis":[int(g.loc[k,"n"]) for k in G],
                       "Weighted %":[round(float(g.loc[k,"pct_weighted"]),1) for k in G]})
    t1.loc[len(t1)] = ["Total", int(ALL.sum()), int(g.n.sum()), round(float(g.pct_weighted.sum()),1)]
    T["Table I: group sizes"] = t1
    T["Table II: settings"] = pd.DataFrame({"Setting":["Records in raw file","Analysis set","Strata","Sampling units","Min count","delta","Holm alpha","R2 gap","Bootstrap","Seed"],
        "Value":["see cell 2 output", s["n_analysis"], s["n_strata"], f'{s["n_psu"]} (one per respondent)', 30, 0.10, 0.05, 0.02, "200 within strata", 2026]})
    c = pd.read_csv(R+"E7_cramers_v.csv")
    T["Table III: Cramer's V"] = pd.DataFrame({"Trait":c.attribute,"Lev.":c.levels,"df*":c.df_star,"V":c.V_unweighted.round(3),
        "V_w":c.V_weighted.round(3),"Size":c.label_weighted,"Deff":c.mean_deff.round(1)})
    pu = pd.read_csv(R+"E3_pruning_unweighted.csv"); pw = pd.read_csv(R+"E3_pruning_weighted.csv")
    T["Table IV: funnel"] = pd.DataFrame({"Step":["Candidates with count >= 30","Holm p<0.05","Holm p<0.05 and Delta>=0.10","After R1 and R2","  of which pairs"],
        "Unweighted":[s["n_candidates_after_min_count"]["1-item"]+s["n_candidates_after_min_count"]["2-item"], cnt["unweighted_holm_pass"],
                      s["n_significant_unweighted"], int(pu.kept.sum()), int((pu.kept & (pu.length==2)).sum())],
        "Weighted":[s["n_candidates_after_min_count"]["1-item"]+s["n_candidates_after_min_count"]["2-item"], cnt["weighted_holm_pass"],
                    s["n_significant_weighted"], int(pw.kept.sum()), int((pw.kept & (pw.length==2)).sum())]})
    e8 = pd.read_csv(R+"E8_conditions_by_group.csv").set_index("condition")[G].round(0).astype(int)
    e8.columns = [GS[x] for x in G]; T["Table V: conditions by group (weighted %)"] = e8
    F = lambda x, d: x.map(lambda v: f"{v:.{d}f}")          # same rounding as the paper
    t6 = pd.DataFrame({"Delta":F(fi.delta,3), "Highest":fi.max_group.map(GS), "Lowest":fi.min_group.map(GS),
        "Char.":b.group.map(GS), "D":F(b["diff"],3), "D lo":F(b.diff_lo,3), "D hi":F(b.diff_hi,3),
        "RR":F(b.rr,2), "Found":F(bs.rediscovery_rate,2), "Same":F(bs.same_max_group_rate,2)})
    t6["below 70%"] = np.where(bs.rediscovery_rate.reindex(t6.index) < .7, "dagger", "")
    t6 = t6.reindex(b["diff"].abs().sort_values(ascending=False).index)
    T["Table VI: all 30 final patterns"] = t6
    se = pd.read_csv(R+"E4_sensitivity.csv")
    se["cell"] = se.apply(lambda r: f"{int(r.n_final)} ({int(r.n_final_pairs)}) / {r.jaccard_vs_base:.2f}", axis=1)
    T["Table VII: threshold grid"] = se.pivot(index="delta", columns="min_count", values="cell")
    return T

def text_numbers():
    N = []; add = lambda sec, what, val: N.append((sec, what, val))
    n_all = int(ALL.sum())
    add("Data", "adults with diabetes", n_all)
    import os
    if os.path.exists("results6/E8_conditions_by_group.csv"):
        add("Data", "depression share of the old six-group 'other single' group (%)", round(float(pd.read_csv("results6/E8_conditions_by_group.csv").set_index("condition").loc["Depressive disorder","diabetes_other_single"]),1))
    add("Data", "analysis set", s["n_analysis"]); add("Data", "removed", s["n_dropped"]); add("Data", "removed (%)", round(100*s["n_dropped"]/n_all,1))
    ms = pd.read_csv(R+"E0_missing_share.csv").set_index("index").share_missing
    add("Data", "income missing (%)", round(100*float(ms["income_group"]),1))
    dd = pd.read_csv(R+"E0_dropout_check.csv"); dd["level"] = dd.level.astype(str)
    fem = dd[(dd.variable=="_SEX")&(dd.level.isin(["2","2.0"]))].iloc[0]; a80 = dd[(dd.variable=="age_group")&(dd.level.isin(["13","13.0"]))].iloc[0]
    add("Data", "female: removed vs kept (%)", f"{fem.pct_dropped:.1f} vs {fem.pct_kept:.1f}")
    add("Data", "aged 80+: removed vs kept (%)", f"{a80.pct_dropped:.1f} vs {a80.pct_kept:.1f}")
    add("Data", "missing-code check: max change in gap", round(json.load(open(R+"legacy_vs_corrected.json"))["max_abs_gap_change"],3))
    r = o.loc[("Smoking: Never smoked","diabetes_copd")]; fr = fi.loc["Smoking: Never smoked"]
    add("Method", "worked example: COPD never smoked vs rest, D", f"{r.supp_group:.3f} vs {r.supp_rest:.3f}, D = {r['diff']:.3f}")
    add("Method", "worked example: Diabetes only share, Delta", f"{fr.supp_diabetes_only:.3f}, Delta = {fr.delta:.3f}")
    add("Method", "single values / pairs generated / candidates", f'{s["n_candidates_after_min_count"]["1-item"]} / {s["pairs_generated"]} / {s["n_candidates_after_min_count"]["1-item"]+s["n_candidates_after_min_count"]["2-item"]}')
    rt = json.load(open(R+"runtime_this_pc.json")); add("Method", "time for the 814 design-based tests, this PC (s)", rt["design_tests_s"])
    c = pd.read_csv(R+"E7_cramers_v.csv")
    add("Results A", "design effect range / SE ratio", f"{c.mean_deff.min():.1f} to {c.mean_deff.max():.1f} / {np.sqrt(c.mean_deff.min()):.1f} to {np.sqrt(c.mean_deff.max()):.1f}")
    add("Results A", "largest Cramer p-value", f"{c.p_rao_scott.max():.2e}")
    add("Results B", "gap passes that fail the design-based test", cnt["weighted_delta_pass_fail_holm"])
    pr = s["pruning_reasons_weighted"]; add("Results B", "removed by R1 / R2", f'{pr.get("R1 complement of a kept item",0)} / {pr.get("R2 not surprising given parents",0)}')
    add("Results B", "final: total / singles / pairs", f'{len(f)} / {int((f.length==1).sum())} / {int((f.length==2).sum())}')
    fu = set(pd.read_csv(R+"E3_pruning_unweighted.csv").query("kept").label)
    add("Results B", "unweighted final / in common", f"{len(fu)} / {len(fu & set(f.label))}")
    add("Results C", "CIs for D that exclude 0", int(((b.diff_lo>0)|(b.diff_hi<0)).sum()))
    vc = b.group.value_counts()
    add("Results C", "characteristic group count: 2+ / COPD / Dep.", f'{vc.get("diabetes_multi_2plus",0)} / {vc.get("diabetes_copd",0)} / {vc.get("diabetes_depression",0)}')
    for p, nm in [("Physical activity: Any leisure activity","2+ active D"),("Smoking: Never smoked","2+ never smoked D")]:
        add("Results C", nm, round(float(o.loc[(p,"diabetes_multi_2plus"),"diff"]),3))
    r = o.loc[("General health: Poor","diabetes_multi_2plus")]; add("Results C", "2+ poor health: RR (share vs rest)", f"{r.rr:.2f} ({100*r.supp_group:.1f}% vs {100*r.supp_rest:.1f}%)")
    r = o.loc[("Smoking: Current, daily","diabetes_copd")]
    add("Results C", "COPD daily smoking: share vs rest, RR [CI]", f"{100*r.supp_group:.1f}% vs {100*r.supp_rest:.1f}%, RR {r.rr:.2f} [{r.rr_lo:.2f}, {r.rr_hi:.2f}]")
    add("Results C", "COPD college graduate D", round(float(o.loc[("Education: College graduate","diabetes_copd"),"diff"]),3))
    r = o.loc[("Smoking: Former smoker","diabetes_copd")]; add("Results C", "COPD former smoker D CI", f"{r.diff_lo:.3f} to {r.diff_hi:.3f}")
    add("Results C", "COPD analysis n / CKD analysis n", f'{int((pd.read_csv(R+"E0_group_sizes.csv").set_index("mm_group").n)["diabetes_copd"])} / {int((pd.read_csv(R+"E0_group_sizes.csv").set_index("mm_group").n)["diabetes_ckd"])}')
    r = o.loc[("Poor mental health days: 0 days","diabetes_depression")]
    add("Results C", "Dep. no poor MH days: share vs rest, RR", f"{100*r.supp_group:.1f}% vs {100*r.supp_rest:.1f}%, RR {r.rr:.2f}")
    add("Results C", "patterns without the two outcome traits", s["n_final_without_outcome_attrs"])
    add("Results D", "characteristic = highest / lowest group", f'{s["ovr_group_equals_max_group"]} / {s["ovr_group_equals_min_group"]} of {len(f)}')
    add("Results D", "bootstrap found in all / >=90% / >=70%", f'{int((bs.rediscovery_rate==1).sum())} / {int((bs.rediscovery_rate>=.9).sum())} / {int((bs.rediscovery_rate>=.7).sum())}')
    add("Results D", "patterns below 70%", "; ".join(f"{p} ({v:.3f})" for p, v in bs.rediscovery_rate[bs.rediscovery_rate<.7].items()))
    copd = [l for l in f.label if "diabetes_copd" in (fi.loc[l,"max_group"], fi.loc[l,"min_group"], b.loc[l,"group"])]
    add("Results D", "patterns involving Diab.+COPD: number / lowest found rate", f"{len(copd)} / {bs.rediscovery_rate[copd].min():.3f}")
    add("Results D", "same highest group in >=90% of samples", int((bs.same_max_group_rate>=.9).sum()))
    add("Results D", "no poor MH days: same highest group rate", round(float(bs.loc["Poor mental health days: 0 days","same_max_group_rate"]),3))
    tc = json.load(open(R+"E5_bootstrap_test_comparison.json"))
    add("Results D", "20 samples: Pearson-only passes / final disagreements", f'{tc["pe_not_rs"]} / {tc["final_flips"]}')
    se = pd.read_csv(R+"E4_sensitivity.csv")
    add("Results E", "delta 0.05 finals / delta 0.15 finals / Jaccard range", f'{se[se.delta==.05].n_final.min()} to {se[se.delta==.05].n_final.max()} / {se[se.delta==.15].n_final.min()} to {se[se.delta==.15].n_final.max()} / {se[se.delta!=.1].jaccard_vs_base.min():.2f} to {se[se.delta!=.1].jaccard_vs_base.max():.2f}')
    add("Results E", "age strata finals: under 65 / 65+", f'{s["age_under65_final_after_pruning"]} / {s["age_65plus_final_after_pruning"]}')
    u = pd.read_csv(R+"E9_final_under65.csv").set_index("label"); v = pd.read_csv(R+"E9_final_65plus.csv").set_index("label")
    k = "Poor mental health days: 14-30 days"; add("Results E", "14-30 MH days Delta: under 65 / 65+", f"{u.loc[k,'delta']:.3f} / {v.loc[k,'delta']:.3f}")
    if os.path.exists(R+"compare_6_vs_7_groups.json"):
      c6 = json.load(open(R+"compare_6_vs_7_groups.json"))
      add("Results E", "six groups: found / kept / new / dropped / Jaccard", f'{c6["six_group_final"]} / {c6["kept"]} / {len(c6["new"])} / {len(c6["dropped"])} / {c6["jaccard"]:.2f}')
    add("Results E", "multinomial McFadden R2", round(s["multinomial_mcfadden_r2"],2))
    m = pd.read_csv(R+"E6_multinomial_logit.csv"); singles = f[f.length==1].label; inm = [x for x in singles if x in set(m.term)]
    sig = set(m[(m.lo>1)|(m.hi<1)].term)
    add("Results E", "final single values in model / with OR CI excluding 1", f"{len(inm)} / {sum(x in sig for x in inm)}")
    w = pd.read_csv(R+"E6_wracc_top10.csv")
    add("Results E", "WRAcc top-10 patterns in our set", f"{int(w.pattern.isin(f.label).sum())} of {len(w)}")
    top = w[w['rank']==1]; add("Results E", "WRAcc top pattern is a mental health pattern in", f'{int(top.pattern.str.contains("mental health").sum())} of {len(top)} groups')
    add("Discussion", "highest-share group not the one that stood out", f'{len(f)-s["ovr_group_equals_max_group"]} of {len(f)}')
    return pd.DataFrame(N, columns=["section", "number in the paper", "your run"])


In [ ]:
import importlib, paper_all; importlib.reload(paper_all)
pd.set_option("display.max_colwidth", 200, "display.max_rows", 200, "display.max_columns", 30)
for name, table in paper_all.tables().items():
    print("=" * 8, name, "=" * 8); display(table)

### 10.1 Every number quoted in the text, section by section
Compare the right column with the sentence in the paper. The raw record count (457,670) is printed in Section 2.

In [ ]:
display(paper_all.text_numbers())

## 11. Checks added after review (missing income, null check, 2-condition group)
These three checks are quoted in Section V-E of the paper. They run on the files made in Sections 2 and 3.
- `run_income.py`: keeps adults with a missing income answer as their own level and reruns the whole pipeline (about 5 to 10 minutes).
- `perm_test.py`: shuffles the group labels inside each stratum and reruns the design-based test (about 15 seconds per shuffle, so about 25 minutes for 100 shuffles). Set `PERM_B = 10` for a quick test; the permutation rows will then not match.
- `two_cond_check.py`: checks the 9 'Diab.+2 or more' patterns inside the group with exactly 2 extra conditions (seconds).

In [ ]:
%%writefile run_income.py
"""Missing-data check: keep records with a missing income answer and give them their own level, 'Not given'.
All other traits must still be complete. Same pipeline and thresholds as the main run."""
import sys; sys.path.insert(0, '.')
import pandas as pd
import csm_upgrade as C
G7 = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
d = pd.read_csv("diabetes_all_7g.csv", low_memory=False)
others = [a for a in C.CFG["attrs"] if a != "income_group"]
d = d[d[others].notna().all(axis=1)].copy()
d["income_group"] = d["income_group"].fillna(8)
d.to_csv("analysis_inc.csv", index=False)
print("records:", len(d), "| income 'Not given':", int((d.income_group == 8).sum()))
C.LABELS["income_group"][8] = "Not given"
C.CFG.update(groups=G7, single_condition_groups=G7[1:6])
C.GROUP_NAMES.update({"diabetes_depression": "Diab.+depression", "diabetes_copd": "Diab.+COPD"})
sys.argv = ["csm_upgrade.py", "--data", "analysis_inc.csv", "--full", "diabetes_all_7g.csv", "--out", "results_inc", "--boot", "2"]
C.main()

# Summary of the check (compared with the 30 patterns of the main run)
import json
a = pd.read_csv("results_inc/final_patterns_weighted.csv"); m = pd.read_csv("results7/final_patterns_weighted.csv")
A, M = set(a.label), set(m.label)
o = pd.read_csv("results_inc/E2_one_vs_rest_weighted.csv").set_index(["pattern", "group"])
bs = pd.read_csv("results7/E5_bootstrap_stability_raoscott.csv").set_index("pattern").rediscovery_rate
least5 = set(bs.sort_values().index[:5])
R = dict(n_records=len(d), n_income_not_given=int((d.income_group == 8).sum()), final_patterns=len(A), main_patterns_kept=len(A & M),
         main_patterns_dropped=len(M - A), new_patterns=len(A - M), least_stable5_dropped=len(least5 - A),
         poor_health_2plus_rr=round(float(o.loc[("General health: Poor", "diabetes_multi_2plus")].rr), 2),
         daily_smoking_copd_rr=round(float(o.loc[("Smoking: Current, daily", "diabetes_copd")].rr), 2))
json.dump(R, open("results7/income_check.json", "w"), indent=1); print(R)


In [ ]:
%%writefile perm_test.py
"""Permutation (null) check. Shuffle the group labels among respondents inside each stratum, then rerun the
design-based test on all 814 candidates with the same filters (Holm p < 0.05 and gap >= 0.10).
Usage: python perm_test.py 100"""
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, json, time
import csm_upgrade as C, fast_rs as F
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg = C.CFG; K = len(G); B = int(sys.argv[1])
df = pd.read_csv("analysis_7g.csv", low_memory=False)
for a in cfg["attrs"]: df[a] = df[a].map(C.norm_code)
g = pd.Categorical(df.mm_group, categories=G).codes
cands = C.candidates(C.build_items(df, cfg["attrs"]), 2); cache = {}
M = np.column_stack([C.mask_of(df, c, cache) for c in cands]); M = M[:, M.sum(0) >= 30]
w = df._LLCPWT.values.astype(float); st = df._STSTR.values
pi, sp, nh = F.design_arrays(st, df._PSU.values)
P, rs, p, de = F.rs1_batch(M, g, K, w, pi, sp, nh)
real = int(((C.holm(p) < .05) & (P.max(1) - P.min(1) >= .1)).sum())
rng = np.random.default_rng(2026); strata_idx = [np.where(st == s)[0] for s in np.unique(st)]
out = []; t0 = time.time()
for b in range(B):
    gp = g.copy()
    for ix in strata_idx: gp[ix] = g[rng.permutation(ix)]
    P, rs, p, de = F.rs1_batch(M, gp, K, w, pi, sp, nh)
    pa = C.holm(p) < .05; ga = (P.max(1) - P.min(1)) >= .1
    out.append(dict(holm_pass=int(pa.sum()), gap_pass=int(ga.sum()), both=int((pa & ga).sum())))
    if b % 10 == 0: print(b, round(time.time() - t0), out[-1], flush=True)
r = pd.DataFrame(out)
R = dict(real_both=real, B=B, shuffles_with_any_pattern=int((r.both > 0).sum()), max_both=int(r.both.max()),
         mean_both=float(r.both.mean()), max_holm_pass=int(r.holm_pass.max()), mean_holm_pass=float(r.holm_pass.mean()),
         mean_gap_pass=float(r.gap_pass.mean()))
json.dump(R, open("results7/perm_test.json", "w"), indent=1); r.to_csv("results7/perm_test_runs.csv", index=False); print(R)


In [ ]:
%%writefile two_cond_check.py
"""Check of the 'Diab.+2 or more' group. For every final pattern whose characteristic group is 'Diab.+2 or more',
compare adults with exactly 2 extra conditions, and adults with 3 or more, with all adults in the other six groups.
D = share in the subgroup minus share in the other six groups; 95% CI by Taylor linearization (each respondent is its
own sampling unit within a stratum)."""
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, json
import csm_upgrade as C
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg = C.CFG
df = pd.read_csv("analysis_7g.csv", low_memory=False)
for a in cfg["attrs"]: df[a] = df[a].map(C.norm_code)
f = pd.read_csv("results7/final_patterns_weighted.csv"); b = pd.read_csv("results7/E2_best_group_per_pattern.csv").set_index("pattern")
pats = [l for l in f.label if b.loc[l, "group"] == "diabetes_multi_2plus"]
cache = {}; w = df._LLCPWT.values.astype(float); st = df._STSTR.values
rest = (df.mm_group != "diabetes_multi_2plus").values; multi = (df.mm_group == "diabetes_multi_2plus").values
def d_ci(y, dom):
    Wg, Wr = w[dom].sum(), w[rest].sum(); sg = (w[dom] * y[dom]).sum() / Wg; sr = (w[rest] * y[rest]).sum() / Wr
    z = np.zeros(len(w)); z[dom] = w[dom] * (y[dom] - sg) / Wg; z[rest] = -w[rest] * (y[rest] - sr) / Wr
    var = 0.0
    for s in np.unique(st):
        zz = z[st == s]; n = len(zz)
        if n > 1: var += n / (n - 1) * ((zz - zz.mean()) ** 2).sum()
    d = sg - sr; se = var ** .5; return sg, sr, d, d - 1.96 * se, d + 1.96 * se
rows = []
cands = C.candidates(C.build_items(df, cfg["attrs"]), 2); lab = {C.pattern_label(c): c for c in cands}
for l in pats:
    y = C.mask_of(df, lab[l], cache).astype(float)
    for name, dom in [("all 2+", multi), ("exactly 2", multi & (df.n_conditions == 2).values), ("3 or more", multi & (df.n_conditions >= 3).values)]:
        sg, sr, d, lo, hi = d_ci(y, dom); rows.append(dict(pattern=l, subgroup=name, n=int(dom.sum()), share=sg, rest=sr, D=d, lo=lo, hi=hi))
r = pd.DataFrame(rows); r.to_csv("results7/two_cond_check.csv", index=False)
main = r[r.subgroup == "all 2+"].set_index("pattern"); two = r[r.subgroup == "exactly 2"].set_index("pattern"); three = r[r.subgroup == "3 or more"].set_index("pattern")
same_sign = (np.sign(two.D) == np.sign(main.D)); excl = ((two.lo > 0) | (two.hi < 0))
R = dict(n_patterns=len(pats), n_two=int((multi & (df.n_conditions == 2).values).sum()), n_three_plus=int((multi & (df.n_conditions >= 3).values).sum()),
         two_same_sign=int(same_sign.sum()), two_ci_excludes_0=int(excl.sum()), two_both=int((same_sign & excl).sum()),
         three_same_sign=int((np.sign(three.D) == np.sign(main.D)).sum()),
         min_ratio_two_over_all=float((two.D / main.D).min()), max_ratio_two_over_all=float((two.D / main.D).max()))
json.dump(R, open("results7/two_cond_check.json", "w"), indent=1); print(r.round(3).to_string()); print(R)


In [ ]:
PERM_B = 100   # the paper uses 100 shuffles
run_script('run_income.py')
run_script('two_cond_check.py')
run_script('perm_test.py', PERM_B)

In [ ]:
EXPECTED_NEW = {
 "income_check": {
  "n_records": 55781,
  "n_income_not_given": 7908,
  "final_patterns": 24,
  "main_patterns_kept": 20,
  "main_patterns_dropped": 10,
  "new_patterns": 4,
  "least_stable5_dropped": 4,
  "poor_health_2plus_rr": 3.22,
  "daily_smoking_copd_rr": 2.36
 },
 "two_cond_check": {
  "n_patterns": 9,
  "n_two": 11416,
  "n_three_plus": 8026,
  "two_same_sign": 9,
  "two_ci_excludes_0": 9,
  "two_both": 9,
  "three_same_sign": 9,
  "min_ratio_two_over_all": 0.486,
  "max_ratio_two_over_all": 0.836
 },
 "perm_test": {
  "real_both": 78,
  "B": 100,
  "shuffles_with_any_pattern": 5,
  "max_both": 3,
  "mean_both": 0.07,
  "max_holm_pass": 64,
  "mean_holm_pass": 43.52,
  "mean_gap_pass": 9.1
 }
}
rows = []
for fn, d in EXPECTED_NEW.items():
    got = json.load(open(f"results7/{fn}.json"))
    for k, v in d.items():
        m = got.get(k); ok = m is not None and abs(m - v) <= (0 if isinstance(v, int) else 0.005)
        rows.append((fn + "." + k, v, m, "PASS" if ok else "FAIL"))
V2 = pd.DataFrame(rows, columns=["number", "paper (my run)", "your run", "result"]); display(V2)
print(f"PASS {(V2.result=='PASS').sum()} of {len(V2)}")
if PERM_B != 100: print("Note: PERM_B is not 100, so the perm_test rows are expected to differ.")
print("Paper sentences: 'Missing income', 'Null check' and 'Two or more conditions' in Section V-E.")

## 12. Checks added after the faculty review (final version of the paper)
These checks are quoted in Sections III-C, III-B (Method), V-E (Checks and Results), Table VI (Perm. column), Limitations and Table II of the final paper. They run on the files made in Sections 2, 3 and 11 (`analysis_7g.csv`, `diabetes_all_7g.csv`, `results7/`). Run Sections 2, 3 and 11 first.

| Script | What it does | Paper statement | Run time (one core) |
|---|---|---|---|
| `maxT.py` | Permutation max-statistic test: shuffles group labels inside strata 100 times, keeps the largest of the 814 test statistics per shuffle | 23.7% of null p-values below 0.05; 15 of 30 patterns pass (10 of 18 single values, 5 of 12 pairs); Perm. column of Table VI | about 17 minutes (`PERM_B = 10` for a quick test) |
| `ipw.py` | Reweights the kept adults by age x sex x group completeness and recomputes the gap of all 30 patterns; Kish design effect and effective group sizes | no gap changes by more than 0.009; effective size 79 (COPD) and 103 (CKD) | seconds |
| `calib_by_groups.py` | Null rate with 3, 5 and 7 groups | 7.7%, 12.5%, 24.9% | about 1 minute |
| `r2_sens.py` | Divides the unweighted Breslow-Day statistic of rule R2 by a design effect D and reruns the whole pipeline | D = 2, 3, 5.8 leave 27, 21, 20 patterns (9, 3, 2 pairs); all 18 single-value patterns stay | about 1 to 2 minutes per D |
| `run_cond_unknown.py` | Removes the 3,355 adults (5.1%) with a "don't know", refused or missing answer on any of the 7 condition questions (the code counts such answers as "no condition") and reruns the pipeline | 28 of the 30 patterns remain (32 patterns in all) | about 5 minutes |
| `rcheck_export.py`, `rcheck.R`, `rcheck_full.R`, `rcheck_summary.py` | Same Rao-Scott tests in R's `survey` package (`svychisq`) on the same data | 19% of null p-values below 0.05 in R (one shuffle, 80 patterns) | about 5 minutes; needs R and the `survey` package, otherwise skipped |

**Things you should know before you trust the numbers (these are limits of the method, and the paper says so):**
- The design-based test is too liberal in the null check, mostly because of small effective group sizes. The Perm. column is the stricter test.
- Rule R2 (Breslow-Day) uses unweighted counts. The paper reports the sensitivity above and treats the 12 pairs as the less certain part of the list.
- The thresholds (gap 0.10, minimum count 30, 0.02 gap for R2) are practical choices, not derived. Table VII of the paper shows other values.
- 27.3% of adults with diabetes are removed for missing data (complete-case analysis). For the five condition questions that define the groups, "don't know" or refused counts as "no condition" (5.1% of adults); `run_cond_unknown.py` shows that removing them keeps 28 of the 30 patterns.
- Several sentences of the earlier paper version (six-group comparison, run-time paragraph, age sentence, thresholds overlap numbers, the missing-code check for smoking and age) were removed to keep 6 pages. Sections 4, 5 and 8 still compute those numbers.

In [ ]:
%%writefile maxT.py
"""Permutation max-statistic test (Section V-E, 'Null check', and the Perm. column of Table VI).
1. Compute the Rao-Scott chi-square statistic of all 814 candidates (count >= 30) on the real data.
2. Shuffle the group labels inside each stratum, recompute all 814 statistics, keep the LARGEST one per shuffle.
3. A final pattern passes if its real statistic is above the 95th percentile of the 100 shuffle maxima.
Also records the share of null p-values below 0.05 and 0.001 over all shuffles.
Usage: python maxT.py 100        (about 10 s per shuffle on one core)"""
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, json
import csm_upgrade as C, fast_rs as F
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg = C.CFG; K = len(G); B = int(sys.argv[1])
df = pd.read_csv("analysis_7g.csv", low_memory=False)
for a in cfg["attrs"]: df[a] = df[a].map(C.norm_code)
g = pd.Categorical(df.mm_group, categories=G).codes
cands = C.candidates(C.build_items(df, cfg["attrs"]), 2); cache = {}
M = np.column_stack([C.mask_of(df, c, cache) for c in cands]); keep = M.sum(0) >= 30
M = M[:, keep]; cands = [c for c, k in zip(cands, keep) if k]
w = df._LLCPWT.values.astype(float); st = df._STSTR.values
pi, sp, nh = F.design_arrays(st, df._PSU.values)
P, rs, p, de = F.rs1_batch(M, g, K, w, pi, sp, nh)            # real data
labels = [C.pattern_label(c) for c in cands]; pos = {l: i for i, l in enumerate(labels)}
fin = pd.read_csv("results7/final_patterns_weighted.csv")
fs = np.array([rs[pos[l]] for l in fin.label])                  # real statistic of each final pattern
rng = np.random.default_rng(2026); sidx = [np.where(st == s)[0] for s in np.unique(st)]
mx, f05, f001 = [], [], []
for b in range(B):
    gp = g.copy()
    for ix in sidx: gp[ix] = g[rng.permutation(ix)]
    P2, rs2, p2, _ = F.rs1_batch(M, gp, K, w, pi, sp, nh)
    mx.append(rs2.max()); f05.append(float((p2 < 0.05).mean())); f001.append(float((p2 < 0.001).mean()))
    if b % 10 == 0: print(b, round(mx[-1], 1), flush=True)
mx = np.array(mx); crit = float(np.percentile(mx, 95))
fin["stat"] = fs; fin["p_adj_perm"] = [(1 + (mx >= x).sum()) / (1 + len(mx)) for x in fs]; fin["pass_perm"] = fs > crit
fin["n_conditions_in_pattern"] = fin.label.str.count(" AND ") + 1
fin[["label", "count", "delta", "stat", "p_adj_perm", "pass_perm"]].to_csv("results7/maxT_final30.csv", index=False)
np.save("results7/maxT_null_max.npy", mx)
R = dict(B=B, real_candidates=int(len(rs)), frac_p_lt_05=float(np.mean(f05)), frac_p_lt_001=float(np.mean(f001)),
         crit95=crit, final30_above_crit=int((fs > crit).sum()),
         single_pass=int(fin.pass_perm[fin.n_conditions_in_pattern == 1].sum()), n_single=int((fin.n_conditions_in_pattern == 1).sum()),
         pairs_pass=int(fin.pass_perm[fin.n_conditions_in_pattern == 2].sum()), n_pairs=int((fin.n_conditions_in_pattern == 2).sum()))
print(R); json.dump(R, open("results7/maxT.json", "w"), indent=1)


In [ ]:
%%writefile ipw.py
"""Missing-data check by reweighting (Section III-C) and weight summary (Section V-E).
Complete-case adults are reweighted by 1 / P(complete | age group x sex x group), estimated among ALL adults with
diabetes. Then the gap D (max minus min share over the seven groups) of each of the 30 final patterns is recomputed.
Also writes the Kish design effect and effective sample sizes."""
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, json
import csm_upgrade as C
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg = C.CFG
al = pd.read_csv("diabetes_all_7g.csv", low_memory=False); df = pd.read_csv("analysis_7g.csv", low_memory=False)
al["key"] = al._STSTR.astype(str) + "|" + al._PSU.astype(str); df["key"] = df._STSTR.astype(str) + "|" + df._PSU.astype(str)
assert al.key.is_unique and df.key.is_unique
al["kept"] = al.key.isin(df.key).astype(int)
cell = al.groupby([al.age_group.fillna(0), al._SEX.fillna(0), "mm_group"]).kept.transform("mean")
al["adj"] = al._LLCPWT / cell
df = df.merge(al[["key", "adj"]], on="key", how="left"); assert df.adj.notna().all()
for a in cfg["attrs"]: df[a] = df[a].map(C.norm_code)
g = pd.Categorical(df.mm_group, categories=G).codes
cands = C.candidates(C.build_items(df, cfg["attrs"]), 2); cache = {}
f = pd.read_csv("results7/final_patterns_weighted.csv"); lab = {C.pattern_label(c): c for c in cands}
M = np.column_stack([C.mask_of(df, lab[l], cache) for l in f.label])
def sup(w): return np.array([[(w[g == k] @ M[g == k, j]) / w[g == k].sum() for k in range(7)] for j in range(M.shape[1])])
w0 = df._LLCPWT.values.astype(float); w1 = df.adj.values.astype(float); S0, S1 = sup(w0), sup(w1)
d0 = S0.max(1) - S0.min(1); d1 = S1.max(1) - S1.min(1)
print("max |paper delta - recomputed delta (original weights)|:", float(np.abs(d0 - f.delta.values).max()))
R = dict(max_abs_change_in_delta=float(np.abs(d1 - d0).max()), n_still_ge_010=int((d1 >= 0.10).sum()),
         same_top_group=int((S0.argmax(1) == S1.argmax(1)).sum()), same_bottom_group=int((S0.argmin(1) == S1.argmin(1)).sum()),
         weight_ratio_min=float((w1 / w0).min()), weight_ratio_max=float((w1 / w0).max()), n_patterns=len(f))
json.dump(R, open("results7/reweight_check.json", "w"), indent=1); print(R)
ne = {gn: round(float(w0[(df.mm_group == gn).values].sum() ** 2 / (w0[(df.mm_group == gn).values] ** 2).sum())) for gn in G}
st = df._STSTR.values; _, cnt = np.unique(st, return_counts=True)
W = dict(kish_deff=float(len(w0) * (w0 ** 2).sum() / w0.sum() ** 2), n_eff_all=float(w0.sum() ** 2 / (w0 ** 2).sum()),
         n_eff_by_group=ne, singleton_strata=int((cnt == 1).sum()))
json.dump(W, open("results7/weights_check.json", "w"), indent=1); print(W)


In [ ]:
%%writefile calib_by_groups.py
"""How the over-liberal null rate depends on the number and size of groups (Section V-E, 'Null check').
Shuffle group labels inside strata (seed 11) and report the share of design-based p-values below 0.05.
Runs: all 7 groups (3 shuffles), 5 groups without Diab.+CKD and Diab.+COPD (4 shuffles), the 3 largest groups (4 shuffles)."""
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, json
import csm_upgrade as C, fast_rs as F
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg = C.CFG
df = pd.read_csv("analysis_7g.csv", low_memory=False)
for a in cfg["attrs"]: df[a] = df[a].map(C.norm_code)
cands = C.candidates(C.build_items(df, cfg["attrs"]), 2); cache = {}
M0 = np.column_stack([C.mask_of(df, c, cache) for c in cands])
def run(groups, B, name):
    keep = df.mm_group.isin(groups).values; d = df[keep].reset_index(drop=True); M = M0[keep]; M = M[:, M.sum(0) >= 30]
    g = pd.Categorical(d.mm_group, categories=groups).codes; K = len(groups)
    w = d._LLCPWT.values.astype(float); st = d._STSTR.values; pi, sp, nh = F.design_arrays(st, d._PSU.values)
    rng = np.random.default_rng(11); sidx = [np.where(st == s)[0] for s in np.unique(st)]; ps = []
    for b in range(B):
        gp = g.copy()
        for ix in sidx: gp[ix] = g[rng.permutation(ix)]
        ps.append(F.rs1_batch(M, gp, K, w, pi, sp, nh)[2])
    p = np.concatenate(ps); r = dict(groups=K, shuffles=B, patterns=int(M.shape[1]), rate_p_lt_005=float((p < .05).mean()))
    print(name, r, flush=True); return r
R = {"all_7_groups": run(G, 3, "all 7"),
     "5_groups_drop_CKD_COPD": run([x for x in G if x not in ("diabetes_ckd", "diabetes_copd")], 4, "5 groups"),
     "3_largest_groups": run(["diabetes_only", "diabetes_arthritis", "diabetes_multi_2plus"], 4, "3 largest")}
json.dump(R, open("results7/calibration_by_groups.json", "w"), indent=1)


In [ ]:
%%writefile r2_sens.py
"""Sensitivity of rule R2 (Breslow-Day test) to survey design.
In the paper's pipeline the Breslow-Day statistic uses UNWEIGHTED counts. Here the statistic is divided by an assumed
design effect D before its p-value is computed (chi-square, 6 df), and the whole pipeline is rerun.
D = 1 must reproduce the paper (30 patterns, 12 pairs). Usage: python r2_sens.py D outdir"""
import sys; sys.path.insert(0, '.')
import csm_upgrade as C
from scipy import stats
from statsmodels.stats.contingency_tables import StratifiedTable
D = float(sys.argv[1]); out = sys.argv[2]
orig = StratifiedTable.test_equal_odds
class R: pass
def patched(self, adjust=False):
    r = orig(self, adjust=adjust); s = r.statistic / D; o = R(); o.statistic = s; o.pvalue = stats.chi2.sf(s, 6); o.df = 6; return o
StratifiedTable.test_equal_odds = patched
G7 = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G7, single_condition_groups=G7[1:6])
C.GROUP_NAMES.update({"diabetes_depression": "Diab.+depression", "diabetes_copd": "Diab.+COPD"})
sys.argv = ["csm_upgrade.py", "--data", "analysis_7g.csv", "--full", "diabetes_all_7g.csv", "--out", out, "--boot", "2"]
C.main()


In [ ]:
%%writefile rcheck_export.py
"""Export data for the R cross-check (R 'survey' package, svychisq). Selected: the 30 final patterns and 50 random other
candidates (seed 5), each with the REAL group labels and one within-strata shuffle (seed 5) of the labels.
Also writes the adults removed for missing data (needed for the full-file domain design in rcheck_full.R)."""
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd
import csm_upgrade as C, fast_rs as F
G = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G); cfg = C.CFG; K = 7
df = pd.read_csv("analysis_7g.csv", low_memory=False)
for a in cfg["attrs"]: df[a] = df[a].map(C.norm_code)
g = pd.Categorical(df.mm_group, categories=G).codes
cands = C.candidates(C.build_items(df, cfg["attrs"]), 2); cache = {}
M = np.column_stack([C.mask_of(df, c, cache) for c in cands]); keep = M.sum(0) >= 30; M = M[:, keep]; cands = [c for c, k in zip(cands, keep) if k]
w = df._LLCPWT.values.astype(float); st = df._STSTR.values; pi, sp, nh = F.design_arrays(st, df._PSU.values)
P, rs, p, de = F.rs1_batch(M, g, K, w, pi, sp, nh)
pos = {C.pattern_label(c): i for i, c in enumerate(cands)}
fi = [pos[l] for l in pd.read_csv("results7/final_patterns_weighted.csv").label]
rng = np.random.default_rng(5); others = [int(i) for i in rng.choice([i for i in range(M.shape[1]) if i not in set(fi)], 150, replace=False)][:50]   # 150 drawn, first 50 used
idx = fi + others
sidx = [np.where(st == s)[0] for s in np.unique(st)]; gp = g.copy()
for ix in sidx: gp[ix] = g[rng.permutation(ix)]
P2, rs2, p2, _ = F.rs1_batch(M, gp, K, w, pi, sp, nh)
out = pd.DataFrame({"w": w, "st": st, "g": g, "gp": gp})
for n, j in enumerate(idx): out[f"x{n}"] = M[:, j].astype(int)
out.to_csv("rcheck_data.csv", index=False)
pd.DataFrame({"n": range(len(idx)), "is_final": [1] * 30 + [0] * 50, "py_p_real": p[idx], "py_p_perm": p2[idx]}).to_csv("rcheck_py.csv", index=False)
al = pd.read_csv("diabetes_all_7g.csv", low_memory=False)
al["key"] = al._STSTR.astype(str) + "|" + al._PSU.astype(str); df["key"] = df._STSTR.astype(str) + "|" + df._PSU.astype(str)
al[~al.key.isin(df.key)][["_LLCPWT", "_STSTR"]].rename(columns={"_LLCPWT": "w", "_STSTR": "st"}).to_csv("rcheck_dropped.csv", index=False)
print("exported", len(idx), "patterns,", len(out), "adults,", int((~al.key.isin(df.key)).sum()), "removed adults")


In [ ]:
%%writefile rcheck_summary.py
"""Summary of the R cross-check. Writes results7/R_crosscheck_summary.json and results7/R_crosscheck.csv."""
import pandas as pd, numpy as np, json
py = pd.read_csv("rcheck_py.csv"); R = pd.read_csv("rcheck_R.csv"); Rf = pd.read_csv("rcheck_R_full.csv")
m = py.merge(R, on="n"); fin = m[m.is_final == 1]; mf = m.merge(Rf, on="n")
S = dict(n_patterns=len(m),
 python_null_rate_p_lt_005=float((m.py_p_perm < .05).mean()), R_null_rate_RS1=float((m.R_p_perm_RS1 < .05).mean()), R_null_rate_F=float((m.R_p_perm_F < .05).mean()),
 R_full_file_null_rate_RS1=float((mf.R_full_perm_RS1 < .05).mean()), R_full_file_null_rate_F=float((mf.R_full_perm_F < .05).mean()), n_patterns_full_file=len(mf),
 final30_python_p_lt_1e4=int((fin.py_p_real < 1e-4).sum()), final30_R_RS1_p_lt_1e4=int((fin.R_p_real_RS1 < 1e-4).sum()),
 final30_R_F_p_lt_005_over_814=int((fin.R_p_real_F < 0.05 / 814).sum()))
m.to_csv("results7/R_crosscheck.csv", index=False); json.dump(S, open("results7/R_crosscheck_summary.json", "w"), indent=1); print(S)


In [ ]:
%%writefile rcheck.R
# R cross-check 1: Rao-Scott tests from the 'survey' package on the same data (subset design, each adult its own PSU).
suppressMessages(library(survey)); options(survey.lonely.psu = "adjust")
d <- read.csv("rcheck_data.csv"); d$gf <- factor(d$g); d$gpf <- factor(d$gp)
n <- 80
for (i in 0:(n - 1)) d[[paste0("f", i)]] <- factor(d[[paste0("x", i)]])
des <- svydesign(ids = ~1, strata = ~st, weights = ~w, data = d)
res <- data.frame(n = 0:(n - 1), R_p_real_RS1 = NA, R_p_real_F = NA, R_p_perm_RS1 = NA, R_p_perm_F = NA)
for (i in 0:(n - 1)) {
  fx <- as.formula(paste0("~f", i, "+gf")); fp <- as.formula(paste0("~f", i, "+gpf"))
  res$R_p_real_RS1[i + 1] <- svychisq(fx, des, statistic = "Chisq")$p.value
  res$R_p_real_F[i + 1]   <- svychisq(fx, des, statistic = "F")$p.value
  res$R_p_perm_RS1[i + 1] <- svychisq(fp, des, statistic = "Chisq")$p.value
  res$R_p_perm_F[i + 1]   <- svychisq(fp, des, statistic = "F")$p.value
  if (i %% 10 == 9) { write.csv(res[1:(i + 1), ], "rcheck_R.csv", row.names = FALSE); cat(i + 1, "of", n, "\n") }
}


In [ ]:
%%writefile rcheck_full.R
# R cross-check 2: same shuffled data, but the variance uses the FULL file (domain estimation: removed adults stay in the design).
suppressMessages(library(survey)); options(survey.lonely.psu = "adjust")
d <- read.csv("rcheck_data.csv"); dr <- read.csv("rcheck_dropped.csv")
n <- 40; cols <- paste0("x", 0:(n - 1))
dr$gp <- NA; for (c in cols) dr[[c]] <- 0
a <- rbind(d[, c("w", "st", "gp", cols)], dr[, c("w", "st", "gp", cols)])
a$gpf <- factor(a$gp); for (c in cols) a[[paste0("f", c)]] <- factor(a[[c]])
des <- svydesign(ids = ~1, strata = ~st, weights = ~w, data = a); sub <- subset(des, !is.na(gp))
res <- data.frame(n = 0:(n - 1), R_full_perm_RS1 = NA, R_full_perm_F = NA)
for (i in 0:(n - 1)) {
  fp <- as.formula(paste0("~fx", i, "+gpf"))
  res$R_full_perm_RS1[i + 1] <- svychisq(fp, sub, statistic = "Chisq")$p.value
  res$R_full_perm_F[i + 1]   <- svychisq(fp, sub, statistic = "F")$p.value
  if (i %% 10 == 9) { write.csv(res[1:(i + 1), ], "rcheck_R_full.csv", row.names = FALSE); cat(i + 1, "of", n, "\n") }
}


In [ ]:
%%writefile run_cond_unknown.py
"""Check on the group definition. In build_data.py a condition counts only if the answer is 'yes' (code 1), so
'don't know', 'refused' and missing answers on the 7 condition questions count as 'no'. This affects 3,355 of the 65,809
adults with diabetes (5.1%). Here those adults are removed from the analysis (they stay in the full file for the design
variance), and the whole pipeline is rerun with the same thresholds. Usage: python run_cond_unknown.py RAW_CSV"""
import sys; sys.path.insert(0, '.')
import pandas as pd, json
import csm_upgrade as C
RAW = sys.argv[1]
CV = ["CVDINFR4", "CVDCRHD4", "CVDSTRK3", "CHCKDNY2", "HAVARTH4", "CHCCOPD3", "ADDEPEV3"]
raw = pd.read_csv(RAW, usecols=["_PSU", "_STSTR", "DIABETE4"] + CV, low_memory=False); raw = raw[raw.DIABETE4 == 1].copy()
raw["unknown"] = (~raw[CV].isin([1, 2]).all(axis=1)).astype(int)
raw["key"] = raw._STSTR.astype(str) + "|" + raw._PSU.astype(str)
a = pd.read_csv("analysis_7g.csv", low_memory=False); a["key"] = a._STSTR.astype(str) + "|" + a._PSU.astype(str)
assert raw.key.is_unique and a.key.isin(raw.key).all()
unk = set(raw.key[raw.unknown == 1])
d = a[~a.key.isin(unk)].drop(columns="key"); d.to_csv("analysis_cu.csv", index=False)
print("adults with diabetes:", len(raw), "| with an unknown condition answer:", int(raw.unknown.sum()), "| in the analysis set:", int(a.key.isin(unk).sum()), "| analysis set now:", len(d))
G7 = ["diabetes_only","diabetes_cvd","diabetes_ckd","diabetes_arthritis","diabetes_depression","diabetes_copd","diabetes_multi_2plus"]
C.CFG.update(groups=G7, single_condition_groups=G7[1:6])
C.GROUP_NAMES.update({"diabetes_depression": "Diab.+depression", "diabetes_copd": "Diab.+COPD"})
sys.argv = ["csm_upgrade.py", "--data", "analysis_cu.csv", "--full", "diabetes_all_7g.csv", "--out", "results_cu", "--boot", "2"]
C.main()
n = pd.read_csv("results_cu/final_patterns_weighted.csv"); m = pd.read_csv("results7/final_patterns_weighted.csv")
A, M = set(n.label), set(m.label)
R = dict(n_diabetes=len(raw), n_unknown_condition=int(raw.unknown.sum()), pct_unknown=round(100 * raw.unknown.mean(), 2), n_records_analysis=len(d),
         final_patterns=len(A), main_patterns_kept=len(A & M), main_patterns_dropped=len(M - A), new_patterns=len(A - M),
         group_sizes=d.mm_group.value_counts().to_dict())
json.dump(R, open("results7/cond_unknown_check.json", "w"), indent=1); print(R)


In [ ]:
PERM_B = 100   # the paper uses 100 shuffles (about 17 minutes). Use 10 for a quick test.
run_script('maxT.py', PERM_B)
run_script('ipw.py')
run_script('calib_by_groups.py')
run_script('run_cond_unknown.py', RAW_CSV)   # about 5 minutes

In [ ]:
# R2 sensitivity: rerun the whole pipeline with the Breslow-Day statistic divided by D. D = 1 must reproduce the paper.
import json, os, pandas as pd
R2 = {}
for D in ["1", "2", "3", "5.8"]:
    run_script("r2_sens.py", D, f"r2_d{D}")
    d = pd.read_csv(f"r2_d{D}/E3_pruning_weighted.csv")
    R2[D] = dict(patterns=int(d.kept.sum()), pairs=int(d[d.length == 2].kept.sum()), singles=int(d[d.length == 1].kept.sum()))
json.dump(R2, open("results7/r2_sensitivity.json", "w"), indent=1); print(R2)

In [ ]:
# Cross-check in R (optional). Needs R on your PATH and the R package 'survey' (install.packages("survey")).
import shutil, subprocess
if shutil.which("Rscript") is None:
    print("Rscript not found. Skipping the R cross-check; the rows for it will show SKIPPED below.")
else:
    ok = subprocess.run(["Rscript", "-e", "library(survey)"], capture_output=True, text=True).returncode == 0
    if not ok: print("R package 'survey' is not installed. Skipping the R cross-check.")
    else:
        run_script("rcheck_export.py")
        for s in ["rcheck.R", "rcheck_full.R"]:
            t0 = time.time(); r = subprocess.run(["Rscript", s], capture_output=True, text=True); print(s, "finished in", round(time.time() - t0), "s"); 
            if r.returncode != 0: print(r.stderr[-800:]); raise RuntimeError(s + " failed")
        run_script("rcheck_summary.py")

In [ ]:
EXPECTED_12 = {
 "maxT": {
  "frac_p_lt_05": 0.237,
  "final30_above_crit": 15,
  "single_pass": 10,
  "n_single": 18,
  "pairs_pass": 5,
  "n_pairs": 12
 },
 "calibration_by_groups": {
  "all_7_groups": 0.249,
  "5_groups_drop_CKD_COPD": 0.125,
  "3_largest_groups": 0.077
 },
 "reweight_check": {
  "max_abs_change_in_delta": 0.009,
  "n_still_ge_010": 28,
  "same_top_group": 30,
  "same_bottom_group": 30
 },
 "r2_sensitivity": {
  "1": [
   30,
   12,
   18
  ],
  "2": [
   27,
   9,
   18
  ],
  "3": [
   21,
   3,
   18
  ],
  "5.8": [
   20,
   2,
   18
  ]
 },
 "cond_unknown_check": {
  "n_unknown_condition": 3355,
  "pct_unknown": 5.1,
  "final_patterns": 32,
  "main_patterns_kept": 28,
  "main_patterns_dropped": 2,
  "new_patterns": 4
 },
 "R_crosscheck_summary": {
  "R_null_rate_RS1": 0.19,
  "R_null_rate_F": 0.19,
  "final30_R_RS1_p_lt_1e4": 30
 }
}
EXPECTED_PERM_COLUMN = {'Age: 70-74': False,
 'Age: 75-79': True,
 'Age: 80+': True,
 'BMI: Obese': False,
 'Education: College graduate': False,
 'General health: Fair': True,
 'General health: Good': False,
 'General health: Good AND BMI: Obese': False,
 'General health: Good AND Education: College graduate': False,
 'General health: Good AND Physical activity: Any leisure activity': False,
 'General health: Poor': True,
 'General health: Very good': True,
 'Income: $100k-200k': False,
 'Physical activity: Any leisure activity': True,
 'Physical activity: Any leisure activity AND Education: College graduate': False,
 'Poor mental health days: 0 days': True,
 'Poor mental health days: 0 days AND General health: Fair': False,
 'Poor mental health days: 0 days AND General health: Good': True,
 'Poor mental health days: 0 days AND Income: $50k-100k': True,
 'Poor mental health days: 0 days AND Physical activity: Any leisure activity': True,
 'Poor mental health days: 0 days AND Physical activity: No leisure activity': False,
 'Poor mental health days: 1-5 days': False,
 'Poor mental health days: 14-30 days': True,
 'Poor mental health days: 14-30 days AND General health: Good': False,
 'Poor mental health days: 14-30 days AND General health: Poor': True,
 'Poor mental health days: 6-13 days': False,
 'Smoking: Current, daily': False,
 'Smoking: Former smoker': True,
 'Smoking: Never smoked': True,
 'Smoking: Never smoked AND Education: College graduate': True}

rows = []
def add(name, want, got, tol):
    if got is None: rows.append((name, want, "n/a", "SKIPPED")); return
    ok = abs(got - want) <= tol; rows.append((name, want, round(got, 4) if isinstance(got, float) else got, "PASS" if ok else "FAIL"))
mt = json.load(open("results7/maxT.json"))
for k, v in EXPECTED_12["maxT"].items(): add("maxT." + k, v, mt[k], 0.0005 if isinstance(v, float) else 0)
cb = json.load(open("results7/calibration_by_groups.json"))
for k, v in EXPECTED_12["calibration_by_groups"].items(): add("null rate, " + k, v, cb[k]["rate_p_lt_005"], 0.0005)
rw = json.load(open("results7/reweight_check.json"))
for k, v in EXPECTED_12["reweight_check"].items(): add("reweight." + k, v, rw[k], 0.0005 if isinstance(v, float) else 0)
wc = json.load(open("results7/weights_check.json"))
add("effective size Diab.+COPD", 79, wc["n_eff_by_group"]["diabetes_copd"], 0); add("effective size Diab.+CKD", 103, wc["n_eff_by_group"]["diabetes_ckd"], 0)
r2 = json.load(open("results7/r2_sensitivity.json"))
for D, (pt, pr, sg) in EXPECTED_12["r2_sensitivity"].items():
    add(f"R2 with D={D}: patterns", pt, r2[D]["patterns"], 0); add(f"R2 with D={D}: pairs", pr, r2[D]["pairs"], 0); add(f"R2 with D={D}: single values", sg, r2[D]["singles"], 0)
cu = json.load(open("results7/cond_unknown_check.json"))
for k, v in EXPECTED_12["cond_unknown_check"].items(): add("condition-unknown check." + k, v, cu[k], 0.005 if isinstance(v, float) else 0)
rs = json.load(open("results7/R_crosscheck_summary.json")) if os.path.exists("results7/R_crosscheck_summary.json") else {}
for k, v in EXPECTED_12["R_crosscheck_summary"].items(): add("R cross-check." + k, v, rs.get(k), 0.005 if isinstance(v, float) else 0)
fin = pd.read_csv("results7/maxT_final30.csv")
bad = [l for l, p in zip(fin.label, fin.pass_perm) if EXPECTED_PERM_COLUMN.get(l) != bool(p)]
rows.append(("Perm. column of Table VI (30 rows)", "30 match", f"{30 - len(bad)} match", "PASS" if not bad else "FAIL"))
V12 = pd.DataFrame(rows, columns=["number", "paper (my run)", "your run", "result"]); display(V12)
print(f"PASS {(V12.result=='PASS').sum()} | FAIL {(V12.result=='FAIL').sum()} | SKIPPED {(V12.result=='SKIPPED').sum()} of {len(V12)}")
if PERM_B != 100: print("Note: PERM_B is not 100, so the maxT rows are expected to differ.")
print("Rows to compare with Table VI (Perm. column):"); display(fin[["label", "delta", "stat", "p_adj_perm", "pass_perm"]].round(3))